# Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities
### A reproducible research notebook · MScFE 690 · Group 17838
**Israel Olanrewaju Odeajo · Uchenna Splendor Ejike · Goodness Chibueze Kalu**

**Research question.** Do stop-only or combined stop-loss/profit-target exits improve net returns or downside risk relative to a fixed-time exit when the entry opportunities are held constant? Does the answer survive costs, entry delay and the price orderings that minute candles cannot resolve?

This notebook contains the data preparation, equations, implementation, checks, experiment, figures, inference and export code. It does **not** clone a repository or call hidden research scripts. Read the narrative cells and run the code from top to bottom in a fresh CPU runtime. The cells are deliberately separated so a reviewer can inspect an intermediate result before continuing.

**Evidence boundary.** The study concerns five fixed equities, not a representative NSE universe. The previous Q1 2021 pilot is development evidence. A working implementation is not evidence of executable profitability or a guarantee of research quality. New market results appear only after the corresponding cells execute successfully; the final test is locked until a matching protocol has been frozen.

| Quantity | What it answers | What it does not answer |
|---|---|---|
| Paired independent-session P&L / reference capital | Effect of changing exits on common, fully evaluable days | Continuous funding or performance on omitted dates |
| Funded, fixed-notional marked equity | Feasibility, sampled drawdown and failure under the declared account rules | Actual fills, unlimited capital, or a stitched path across unknown losses |
| Differences between candle paths | Sensitivity to the two declared within-minute orderings | Recovered tick order or universal portfolio bounds |
| ES97.5 of daily losses | Observed tail severity for each policy | Precise population tail risk from a few extreme days |


## 1. How to use this notebook

1. Save a copy in your Google Drive. Add shortcuts to the instructor's **NSE Cash Stocks Data** and **NSE Index Data** folders under **My Drive**. The instructor's sharing permissions must allow your account to read the files.
2. Choose a stage below. `pilot` reproduces the previously examined Q1 2021 development period. `development` and `validation` use the fixed calendar partitions of the 2021–2023 study. `final` requires the saved validation freeze.
3. Run in a **hosted CPU runtime**. The loader streams selected CSVs into a disk-backed database and the strategy reads one session at a time. Drive access still uses the hosted runtime's memory and disk; it does not make computation memory-free. Source archives are never altered.
4. Inspect the data audit, synthetic checks and exclusions before interpreting the results. Read warnings about unidentified outcomes and unsupported intervals.
5. Save the generated run folder. It contains tables, figures, a compact HTML report, the protocol and a machine-readable audit. Source prices and detailed ledgers stay in the private working directory.

The default is **development**, not the final test. Do not change a threshold, stock, cost or exclusion after looking at final-test performance and still call that period untouched. If a final run is repeated, its previous access is retained in the access log.


In [ ]:
# Colab form controls: change paths/stage here, not the research rules below.
RUN_STAGE = "development" # @param ["pilot", "development", "validation", "final"]
CONNECT_DRIVE = True # @param {type:"boolean"}
CASH_FOLDER = "/content/drive/MyDrive/NSE Cash Stocks Data" # @param {type:"string"}
INDEX_FOLDER = "/content/drive/MyDrive/NSE Index Data" # @param {type:"string"}
SAVE_FOLDER = "/content/drive/MyDrive/WQU_Capstone_17838/research" # @param {type:"string"}

from pathlib import Path
from datetime import date, datetime, timedelta, timezone
import csv, hashlib, inspect, io, json, math, os, platform, shutil, sqlite3, subprocess, sys, tempfile, time as clock
for variable in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[variable] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

if RUN_STAGE not in {"pilot", "development", "validation", "final"}:
    raise ValueError("Choose one of the four declared stages")
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
if CONNECT_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook in Colab, or explicitly configure local verification paths")
    from google.colab import drive
    drive.mount('/content/drive')
ROOTS = [Path(CASH_FOLDER), Path(INDEX_FOLDER)]
SAVE_BASE = Path(SAVE_FOLDER)
SAVE_BASE.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(tempfile.mkdtemp(prefix='nse-research-', dir='/content' if IN_COLAB else None))
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + WORK_DIR.name.rsplit('-',1)[-1]
OUTPUT_DIR = WORK_DIR / 'outputs'
OUTPUT_DIR.mkdir()
# Local quality checks may supply a previously audited small database. Never a silent fallback.
EXISTING_DB_PATH = globals().get('EXISTING_DB_PATH', None)
if not EXISTING_DB_PATH and not shutil.which('bsdtar') and sys.platform != 'win32':
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libarchive-tools'], check=True)
plt.rcParams.update({'figure.dpi':115, 'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.grid':True, 'grid.alpha':0.18, 'font.size':10})
print('Stage:', RUN_STAGE, '| Runtime:', platform.python_version(), platform.system())
print('Private working directory:', WORK_DIR)


## 2. Research design and the frozen comparison

Stops can remove a position before the recovery a reversal strategy seeks. Kaminski and Lo show why the return dynamics matter; their model does not establish that a stop helps this strategy. Leung and Li derive interacting stop and liquidation boundaries in a specified mean-reverting model; our fixed barriers are **not** claimed to be optimal (Kaminski and Lo 242–44; Leung and Li 15).

| Component | Declared choice | Reason / boundary |
|---|---|---|
| Universe | RELIANCE, HDFCBANK, INFY, TCS, ITC | Same fixed five-stock case study as the proposal; not selected by profit |
| Benchmark | `.NSEI`, used only in the signal | Coefficient-one index adjustment; not an estimated beta or a traded hedge |
| Entry | Long when same-clock index-adjusted z ≤ −2 | One signal; no threshold search |
| Exits | Time-only; stop-only; stop plus target | Same entries, notionals and 15-minute expiry within a delay scenario |
| Capital | C₀ = INR 100,000; each episode 0.5C₀ / 5 | Fixed fractional-share notionals; no compounding or borrowing |
| Reference execution | 15 bp round trip; one-minute delay; zero extra stop penalty | Declared research scenarios, not reconstructed historical fees |
| Paths | O-L-H-C and O-H-L-C | Conditional orderings, not observed ticks |
| Timestamp interpretation | Both minute-start and minute-end | Supplier convention remains unconfirmed; never choose by returns |
| Primary contrast | Combined minus time-only mean daily P&L / C₀ | Complete-case dates, reference execution, O-L-H-C |
| Secondary contrasts | Stop minus time; combined minus stop | The latter adds a target conditional on a stop; no target-only effect is identified |

The full study extends the submitted design rather than changing the topic. It adds funded accounting and a specified resampler that were unfinished in the M4 pilot. These are prospective implementation additions, not previously submitted results. No machine-learning model, optimisation sweep or unrelated strategy is introduced.


In [ ]:
STUDY_START, STUDY_END = '2021-01-01', '2023-12-31'
CFG = dict(study_title='Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities',
           start_date=STUDY_START, end_date=STUDY_END,
           universe=['RELIANCE','HDFCBANK','INFY','TCS','ITC'], index_symbol='.NSEI',
           reference_capital_inr=100000.0, allocation_fraction=0.5,
           prior_sessions=20, signal_window_minutes=15, z_threshold=-2.0,
           decision_minutes_after_open=30, decision_interval_minutes=30, holding_minutes=15,
           normal_session_open='09:15', normal_session_close='15:30',
           excluded_nonstandard_sessions=['2021-02-24'], funded_restart_dates={})
SETTINGS = [dict(name='reference',cost_bp=15.0,stop_slippage_bp=0.0,delay_minutes=1),
            dict(name='cost_5',cost_bp=5.0,stop_slippage_bp=0.0,delay_minutes=1),
            dict(name='cost_25',cost_bp=25.0,stop_slippage_bp=0.0,delay_minutes=1),
            dict(name='stop_slippage_5',cost_bp=15.0,stop_slippage_bp=5.0,delay_minutes=1),
            dict(name='stop_slippage_10',cost_bp=15.0,stop_slippage_bp=10.0,delay_minutes=1),
            dict(name='delay_2',cost_bp=15.0,stop_slippage_bp=0.0,delay_minutes=2),
            dict(name='delay_5',cost_bp=15.0,stop_slippage_bp=0.0,delay_minutes=5)]
INFERENCE = dict(confidence=0.95, repetitions=2000, seed=17838,
                 block_lengths=[5,10,20], minimum_block_support=10,
                 short_segment_rule='withhold interval, retain all days in point estimate',
                 method='paired moving blocks separately within contiguous eligible calendar segments')
if RUN_STAGE == 'pilot':
    CFG['end_date'] = '2021-03-31'
display(pd.DataFrame([{'parameter':key, 'value':str(value)} for key,value in CFG.items()]))
display(pd.DataFrame(SETTINGS))


## 3. Calendar, chronological partitions and past-only information

The split is based on the **scheduled normal-session calendar**, before data or strategy exclusions: development ends at floor(0.60n), validation at floor(0.80n), and final test is the remainder. Each stage has its own initial funded capital. Historical observations may initialise the next stage's features; positions never cross a day or partition boundary.

The holiday lists below come from NSE capital-market circulars [CMTR46623](https://archives.nseindia.com/content/circulars/CMTR46623.pdf), [CMTR50560](https://archives.nseindia.com/content/circulars/CMTR50560.pdf) and [CMTR54757](https://archives.nseindia.com/content/circulars/CMTR54757.pdf). The 2023 Bakri Id holiday is corrected to **29 June**, as recorded in NSE's [27 June amendment cross-reference](https://archives.nseindia.com/content/circulars/CMPT57291.pdf). Normal-session scope excludes Muhurat sessions and weekends; it is not a claim that the exchange had no special sessions. Observed off-calendar records are reported rather than used to invent a calendar.

The known **24 February 2021** outage remains a scheduled date with unavailable features and no simulated trades. Retaining it as missing history affects the next twenty sessions. This preserves the pilot restriction; it is not an estimate of outage-day losses or a claim that every price that day is unusable. See the [SEBI order](https://www.sebi.gov.in/sebi_data/attachdocs/jun-2023/1687270559560.pdf). Calendar amendments require an explicit protocol change, not a change prompted by attractive results.


In [ ]:
HOLIDAYS = {
 2021: '01-26 03-11 03-29 04-02 04-14 04-21 05-13 07-21 08-19 09-10 10-15 11-04 11-05 11-19',
 2022: '01-26 03-01 03-18 04-14 04-15 05-03 08-09 08-15 08-31 10-05 10-24 10-26 11-08',
 2023: '01-26 03-07 03-30 04-04 04-07 04-14 05-01 06-29 08-15 09-19 10-02 10-24 11-14 11-27 12-25'}
holiday_dates = {date.fromisoformat(f'{year}-{day}') for year,days in HOLIDAYS.items() for day in days.split()}
ALL_CALENDAR = [d.date() for d in pd.date_range(STUDY_START, STUDY_END)
                if d.weekday()<5 and d.date() not in holiday_dates]
n = len(ALL_CALENDAR)
i60, i80 = math.floor(.60*n), math.floor(.80*n)
PARTITIONS = {'development':ALL_CALENDAR[:i60], 'validation':ALL_CALENDAR[i60:i80],
              'final':ALL_CALENDAR[i80:]}
assert sum(map(len, PARTITIONS.values())) == n
assert max(PARTITIONS['development']) < min(PARTITIONS['validation']) < min(PARTITIONS['final'])
assert date(2023,6,28) in ALL_CALENDAR and date(2023,6,29) not in ALL_CALENDAR
CALENDAR = [d for d in ALL_CALENDAR if CFG['start_date'] <= d.isoformat() <= CFG['end_date']]
EVALUATION_DATES = CALENDAR[CFG['prior_sessions']:] if RUN_STAGE=='pilot' else PARTITIONS[RUN_STAGE]
split_table = pd.DataFrame([dict(stage=k,start=v[0],end=v[-1],scheduled_sessions=len(v))
                           for k,v in PARTITIONS.items()])
display(split_table)
print('Current stage:', EVALUATION_DATES[0], 'to', EVALUATION_DATES[-1],
      '| Final-test strategy outcomes remain gated below.')
fig, ax = plt.subplots(figsize=(9,1.9))
for j,(name,days) in enumerate(PARTITIONS.items()):
    ax.plot([pd.Timestamp(days[0]),pd.Timestamp(days[-1])],[j,j],lw=11,solid_capstyle='butt',label=name)
ax.set(yticks=range(3), yticklabels=list(PARTITIONS), xlabel='Scheduled calendar date',title='Partitions fixed before exclusions')
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'calendar_split.png'); plt.show()


## Read the instructor data without loading it all into memory

`ROOTS` lists the mounted **NSE Cash Stocks Data** and **NSE Index Data** folders.
The loader searches their subfolders for exact monthly archive names. It also
accepts extracted CSVs if their original monthly folder names are preserved.
For example, `Cash Data January 2021/RELIANCE.csv` is a supported extracted file.
The yearly `NSE-Index-2021.7z` container is supported, as are the monthly index
RAR files supplied for 2022–2023. Outer ZIP download bundles are not used here.

Only the configured five stocks and `.NSEI` are read. Each selected archive is
copied to temporary Colab disk, fingerprinted with SHA-256, and discarded after its
selected CSVs have been streamed into SQLite. Missing sources and ambiguous
copies stop the run. The June 2023 missing-data supplement is listed separately;
it is **not merged** without a documented conflict and provenance rule.

Dates use explicit `MM/DD/YYYY` or `DD-MM-YYYY` formats; the August 2023 cash
files switch to the latter. No date-order guessing is used, and every row must
match its declared source month, apart from the explicitly verified copies below. Prices must be finite, positive and consistent with
the candle range. Within-month duplicate or backward timestamps and malformed rows stop ingestion.
The September 2023 cash files include copies of August 22. An out-of-month row
is excluded only if its timestamp and all five OHLCV values exactly match an
already validated row from the proper monthly source; otherwise ingestion stops.
The audit counts these exclusions. September 22 stays missing; no dates are relabelled.
There is no forward-fill, silent row removal, price adjustment or inferred
trading calendar. Raw minute labels are preserved: neither parsing nor a complete
file proves their timezone or whether they label the start or end of a minute.
The optional ninth vendor column is not interpreted or stored in `bars`; the
audit explicitly records it as unused, and the source fingerprint covers its
original file or archive. No open-interest meaning is assumed.

**Outputs for later cells:** `DB_PATH` and `DB_READONLY_URI`; a `bars` table with
`symbol, stamp, o, h, l, c, volume, source_id`; `DATA_AUDIT`; and `COVERAGE_ROWS`.
The optional `EXISTING_DB_PATH` is only a local smoke-test shortcut. It is clearly
labelled and cannot establish that raw Drive inputs have been verified.


In [ ]:
import calendar, csv, hashlib, io, json, math, re, shutil, sqlite3, subprocess, sys, tempfile
from collections import defaultdict
from contextlib import closing, contextmanager
from datetime import date, datetime
from pathlib import Path

def month_keys(start, end):
    first, last = date.fromisoformat(start), date.fromisoformat(end)
    if first > last:
        raise ValueError('start_date must be on or before end_date')
    year, month = first.year, first.month
    result = []
    while (year, month) <= (last.year, last.month):
        result.append(f'{year:04d}-{month:02d}')
        year, month = (year + 1, 1) if month == 12 else (year, month + 1)
    return result

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def archive_command(*args):
    executable = shutil.which('bsdtar') or (shutil.which('tar') if sys.platform == 'win32' else None)
    if executable is None:
        raise RuntimeError('Install libarchive-tools in Colab to provide bsdtar.')
    return [executable, *map(str, args)]

@contextmanager
def archive_lines(path, member=None):
    command = archive_command('-tf', path) if member is None else archive_command('-xOf', path, member)
    with tempfile.TemporaryFile(mode='w+t') as errors:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=errors,
                                   text=True, encoding='utf-8-sig')
        try:
            yield process.stdout
            process.stdout.close()
            if process.wait():
                errors.seek(0)
                raise RuntimeError(errors.read(2000))
        finally:
            if process.poll() is None:
                process.kill()
                process.wait()
            process.stdout.close()

def member_named(names, basename):
    matches = [name for name in names if name.replace('\\', '/').rsplit('/', 1)[-1] == basename]
    if len(matches) != 1:
        raise ValueError(f'Expected one {basename}; found {len(matches)}')
    return matches[0]

def archive_members(path):
    with archive_lines(path) as lines:
        return [line.rstrip('\r\n') for line in lines]

def vendor_rows(lines, symbol, month=None):
    reader = csv.reader(lines, strict=True)
    fields = ('<ticker>', '<date>', '<time>', '<open>', '<high>', '<low>', '<close>', '<volume>')
    header = tuple(item.strip().lstrip('\ufeff') for item in next(reader, []))
    if header != fields and not (len(header) == 9 and header[:8] == fields and header[8] in ('<o/i>', '')):
        raise ValueError('Unsupported/absent header; headerless supplements are not accepted')
    previous = None
    for row in reader:
        try:
            if len(row) != len(header) or row[0].strip() != symbol:
                raise ValueError('Wrong column count or ticker')
            day, clock = row[1].strip(), row[2].strip()
            if re.fullmatch(r'[0-9]{2}/[0-9]{2}/[0-9]{4}', day):
                date_format = '%m/%d/%Y'
            elif re.fullmatch(r'[0-9]{2}-[0-9]{2}-[0-9]{4}', day):
                date_format = '%d-%m-%Y'
            else:
                raise ValueError(f'Expected MM/DD/YYYY or DD-MM-YYYY, received {day!r}')
            if not re.fullmatch(r'[0-9]{2}:[0-9]{2}:00', clock):
                raise ValueError(f'Expected minute-boundary HH:MM:00, received {clock!r}')
            stamp = datetime.strptime(day + ' ' + clock, date_format + ' %H:%M:%S')
            o, h, l, c, volume = map(float, row[3:8])
            if not all(math.isfinite(x) and x > 0 for x in (o, h, l, c)) or not l <= min(o, c) <= max(o, c) <= h:
                raise ValueError('Invalid OHLC prices/range')
            if not math.isfinite(volume) or volume < 0:
                raise ValueError('Invalid volume')
            if month is None or stamp.strftime('%Y-%m') == month:
                if previous is not None and stamp <= previous:
                    raise ValueError('Duplicate or out-of-order timestamp')
                previous = stamp
        except (ValueError, OverflowError) as error:
            raise ValueError(f'{symbol}, CSV line {reader.line_num}: {error}') from error
        yield stamp.isoformat(), o, h, l, c, volume


In [ ]:
def discover_sources(roots, cfg):
    roots = list(dict.fromkeys(Path(root).resolve() for root in roots))
    for root in roots:
        if not root.is_dir():
            raise FileNotFoundError(f'Check the mounted Drive folder: {root}')
    files = sorted({path.resolve() for root in roots for path in root.rglob('*')
                    if path.is_file() and path.suffix.lower() in ('.csv', '.rar', '.7z')})
    by_name = defaultdict(list)
    for path in files:
        by_name[path.name].append(path)
    plan, missing = [], []
    for ym in month_keys(cfg['start_date'], cfg['end_date']):
        year, number = map(int, ym.split('-'))
        month = calendar.month_name[number]
        for stem, symbols in ((f'Cash Data {month} {year}', cfg['universe']),
                              (f'NSE Indices-{month} {year}', [cfg['index_symbol']])):
            direct = by_name[stem + '.rar']
            yearly = by_name[f'NSE-Index-{year}.7z'] if symbols == [cfg['index_symbol']] else []
            extracted = {symbol: [p for p in by_name[symbol + '.csv'] if stem in [q.name for q in p.parents]]
                         for symbol in symbols}
            has_csv = any(extracted.values())
            choices = len(direct) + len(yearly) + int(has_csv)
            if choices > 1 or any(len(paths) > 1 for paths in extracted.values()):
                raise ValueError(f'Ambiguous copies of {stem}; choose one source root/layout')
            if has_csv:
                for symbol, paths in extracted.items():
                    if not paths:
                        missing.append(f'{ym}: {symbol}.csv in {stem}')
                    else:
                        plan.append(dict(kind='csv', path=paths[0], month=ym, symbols=[symbol]))
            elif direct or yearly:
                plan.append(dict(kind='rar' if direct else 'year', path=(direct or yearly)[0],
                                 month=ym, symbols=list(symbols), member=stem + '.rar'))
            else:
                missing.append(stem + '.rar or extracted CSVs')
    supplements = [str(p) for p in files if 'missing' in p.name.lower()]
    return plan, missing, supplements

# Synthetic checks: these contain no market data and never open a Drive file.
header = '<ticker>,<date>,<time>,<open>,<high>,<low>,<close>,<volume>\n'
row = 'SYNTHETIC,03/04/2021,09:15:00,100,101,99,100.5,250\n'
assert list(vendor_rows(io.StringIO(header + row), 'SYNTHETIC'))[0][0] == '2021-03-04T09:15:00'
assert len(list(vendor_rows(io.StringIO(chr(65279) + header.replace(chr(10), chr(13)+chr(10)) + row), 'SYNTHETIC'))) == 1
for bad in (row + row, row.replace('100,101,99', '100,99,98'), row.replace('09:15:00', '09:15:01')):
    try:
        list(vendor_rows(io.StringIO(header + bad), 'SYNTHETIC'))
    except ValueError:
        pass
    else:
        raise AssertionError('Malformed synthetic data was accepted')
with tempfile.TemporaryDirectory() as demo:
    folder = Path(demo)
    (folder / 'Cash Data January 2021.rar').touch()
    (folder / 'NSE-Index-2021.7z').touch()
    cfg_demo = dict(start_date='2021-01-01', end_date='2021-01-31', universe=['SYNTHETIC'], index_symbol='.NSEI')
    plan_demo, missing_demo, _ = discover_sources([folder], cfg_demo)
    assert len(plan_demo) == 2 and not missing_demo
    (folder / 'Cash Data January 2021').mkdir()
    (folder / 'Cash Data January 2021' / 'SYNTHETIC.csv').touch()
    try:
        discover_sources([folder], cfg_demo)
    except ValueError:
        pass
    else:
        raise AssertionError('Ambiguous synthetic sources were accepted')
print('Synthetic parser and source-discovery checks passed.')


In [ ]:
def build_database(plan, cfg, work_dir):
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    # A fresh attempt preserves any partial database from a failed read.
    database = Path(tempfile.mkdtemp(prefix='input-', dir=work_dir)) / 'bars.sqlite'
    con = sqlite3.connect(database)
    con.execute('PRAGMA cache_size=-2048')
    con.execute('PRAGMA temp_store=FILE')
    con.execute('CREATE TABLE bars(symbol TEXT, stamp TEXT, o REAL,h REAL,l REAL,c REAL,volume REAL,source_id INTEGER,PRIMARY KEY(symbol,stamp)) WITHOUT ROWID')
    manifest = []

    def ingest(lines, symbol, item, provenance):
        source_id, rows, retained, first, last = len(manifest) + 1, 0, 0, None, None
        digest = hashlib.sha256()
        duplicates = 0
        for stamp, o, h, l, c, volume in vendor_rows(lines, symbol, month=item['month']):
            rows += 1
            digest.update(f'{symbol},{stamp},{o!r},{h!r},{l!r},{c!r},{volume!r}\n'.encode())
            if stamp[:7] != item['month']:
                known = con.execute('SELECT o,h,l,c,volume FROM bars WHERE symbol=? AND stamp=?',
                                    (symbol, stamp)).fetchone()
                if known != (o, h, l, c, volume):
                    raise ValueError(f'Unmatched/conflicting out-of-month row: {symbol}, {stamp}')
                duplicates += 1
                continue  # Exact copies of an already validated monthly source; never relabel dates.
            first, last = first or stamp, stamp
            # Boundary months are fully validated; dates outside the requested range are counted.
            if cfg['start_date'] <= stamp[:10] <= cfg['end_date']:
                con.execute('INSERT INTO bars VALUES(?,?,?,?,?,?,?,?)', (symbol, stamp, o, h, l, c, volume, source_id))
                retained += 1
        if not rows or not retained:
            raise ValueError(f'Empty requested sample for {symbol}, {item["month"]}')
        manifest.append(dict(source_id=source_id, symbol=symbol, month=item['month'], rows=rows,
                             retained_rows=retained, outside_requested_dates=rows-retained-duplicates,
                             excluded_exact_out_of_month_duplicates=duplicates,
                             first_label=first, last_label=last, parsed_ohlcv_sha256=digest.hexdigest(), **provenance))
        con.commit()

    grouped = defaultdict(list)
    for item in plan:
        grouped[item['path']].append(item)
    try:
        for source, items in grouped.items():
            if items[0]['kind'] == 'csv':
                provenance = dict(source=str(source), csv_sha256=file_hash(source), optional_ninth_field='not used')
                with source.open(encoding='utf-8-sig', newline='') as lines:
                    ingest(lines, items[0]['symbols'][0], items[0], provenance)
            else:
                with tempfile.TemporaryDirectory(dir=work_dir) as staging:
                    staged = Path(staging) / source.name
                    shutil.copyfile(source, staged)
                    source_hash, names = file_hash(staged), archive_members(staged)
                    for item in items:
                        monthly = staged
                        provenance = dict(source=str(source), archive_sha256=source_hash, optional_ninth_field='not used')
                        if item['kind'] == 'year':
                            nested = member_named(names, item['member'])
                            monthly = Path(staging) / item['member']
                            with monthly.open('wb') as output:
                                subprocess.run(archive_command('-xOf', staged, nested), stdout=output, check=True)
                            provenance.update(year_member=nested, monthly_archive_sha256=file_hash(monthly))
                        monthly_names = names if monthly == staged else archive_members(monthly)
                        for symbol in item['symbols']:
                            member = member_named(monthly_names, symbol + '.csv')
                            with archive_lines(monthly, member) as lines:
                                ingest(lines, symbol, item, {**provenance, 'member': member})
                        if monthly != staged:
                            monthly.unlink()  # Only the temporary copy made immediately above.
            print('Validated:', source.name)
        audit = dict(mode='raw instructor inputs', raw_inputs_reverified=True, input_files=manifest,
                     requested_start=cfg['start_date'], requested_end=cfg['end_date'],
                     assumptions='Raw naive minute labels; timezone and interval convention unverified; no filling or adjustment.')
        (work_dir / 'input_audit.json').write_text(json.dumps(audit, indent=2) + '\n')
        return database, audit
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


In [ ]:
existing = globals().get('EXISTING_DB_PATH')
if existing:
    DB_PATH = Path(existing).resolve()
    if not DB_PATH.is_file():
        raise FileNotFoundError(DB_PATH)
    DATA_AUDIT = dict(mode='EXISTING DATABASE: LOCAL SMOKE TEST ONLY', raw_inputs_reverified=False,
                      database_sha256=file_hash(DB_PATH), requested_start=CFG['start_date'],
                      requested_end=CFG['end_date'], assumptions='Existing raw-label database; source inputs not reverified here.')
    print(DATA_AUDIT['mode'])
else:
    SOURCE_PLAN, MISSING_SOURCES, EXCLUDED_SUPPLEMENTS = discover_sources(ROOTS, CFG)
    print('Requested months:', len(month_keys(CFG['start_date'], CFG['end_date'])))
    print('Selected source entries:', len(SOURCE_PLAN))
    print('Unmerged supplements:', [Path(p).name for p in EXCLUDED_SUPPLEMENTS])
    if MISSING_SOURCES:
        print('Missing requested sources:\n' + '\n'.join(MISSING_SOURCES))
        raise FileNotFoundError('Resolve missing sources or explicitly revise the requested scope before ingestion.')
    DB_PATH, DATA_AUDIT = build_database(SOURCE_PLAN, CFG, WORK_DIR)
    DATA_AUDIT['excluded_supplements'] = EXCLUDED_SUPPLEMENTS
    print('Imported rows:', sum(x['retained_rows'] for x in DATA_AUDIT['input_files']),
          '| Verified out-of-month copies excluded:', sum(x['excluded_exact_out_of_month_duplicates'] for x in DATA_AUDIT['input_files']))
    (Path(WORK_DIR) / 'input_audit.json').write_text(json.dumps(DATA_AUDIT, indent=2) + '\n')
DB_READONLY_URI = DB_PATH.resolve().as_uri() + '?mode=ro'


### Coverage check — availability, not performance

The following table counts observations without displaying prices or returns.
Every requested symbol-month must be present. The day-count table records the
number of raw labels in 09:15–15:29 and labels outside that window. It does not
infer exchange holidays from missing observations, and 375 labels alone do not
prove the correct timestamp convention. Missing minutes remain missing for the
later, calendar-based eligibility checks. Full source details remain in the
private `input_audit.json`; do not publish Drive paths or raw price records.


In [ ]:
with closing(sqlite3.connect(DB_READONLY_URI, uri=True)) as con:
    required_columns = ['symbol', 'stamp', 'o', 'h', 'l', 'c', 'volume', 'source_id']
    if [row[1] for row in con.execute('PRAGMA table_info(bars)')] != required_columns:
        raise ValueError('Unexpected database schema')
    scope = (CFG['start_date'], CFG['end_date'])
    coverage = con.execute("""
        SELECT symbol, substr(stamp,1,7), COUNT(*), COUNT(DISTINCT substr(stamp,1,10)), MIN(stamp), MAX(stamp)
        FROM bars WHERE substr(stamp,1,10) BETWEEN ? AND ?
        GROUP BY symbol, substr(stamp,1,7) ORDER BY 2,1""", scope).fetchall()
    observed = {(row[0], row[1]) for row in coverage}
    expected = {(symbol, ym) for symbol in [*CFG['universe'], CFG['index_symbol']]
                for ym in month_keys(*scope)}
    if expected - observed:
        raise ValueError(f'Database does not cover requested symbol-months: {sorted(expected-observed)}')
    if observed - expected:
        raise ValueError(f'Unexpected symbols in database scope: {sorted(observed-expected)}')
    daily = con.execute("""
        SELECT symbol, substr(stamp,1,10), COUNT(*),
               SUM(substr(stamp,12)>='09:15:00' AND substr(stamp,12)<'15:30:00')
        FROM bars WHERE substr(stamp,1,10) BETWEEN ? AND ?
        GROUP BY symbol, substr(stamp,1,10) ORDER BY 2,1""", scope).fetchall()
COVERAGE_ROWS = [dict(zip(('symbol', 'month', 'rows', 'observed_days', 'first_label', 'last_label'), row)) for row in coverage]
RAW_LABEL_DAY_COUNTS = [dict(symbol=symbol, date=day, rows=rows, raw_0915_1529_labels=regular,
                             outside_window_labels=rows-regular) for symbol, day, rows, regular in daily]
DATA_AUDIT['coverage'] = COVERAGE_ROWS
DATA_AUDIT['raw_label_day_counts'] = RAW_LABEL_DAY_COUNTS
if not globals().get('EXISTING_DB_PATH'):
    (Path(WORK_DIR) / 'input_audit.json').write_text(json.dumps(DATA_AUDIT, indent=2) + '\n')
print('Symbol       Month      Rows  Observed days')
for row in COVERAGE_ROWS:
    print(f'{row["symbol"]:12} {row["month"]} {row["rows"]:7,d} {row["observed_days"]:14}')
print('Symbol-days with other than 375 raw labels in 09:15–15:29:', sum(row['raw_0915_1529_labels'] != 375 for row in RAW_LABEL_DAY_COUNTS))
print('No price, return, holdout result or inferred trading calendar was displayed.')


### Coverage is an input to interpretation

A month being present does not make every minute usable. The heatmap compares the two timestamp interpretations with 375 normal-session minute labels, including missing days as zero observed labels. These are **coverage counts**, not strategy returns. A thin month or a special session cannot silently disappear from the denominator. Volume can describe activity but does not reveal bid–ask spreads, executable depth or queue priority.

The supplied adjustment convention, corporate-action treatment and tick/lot rules remain unverified. Intraday-only exposure avoids an overnight position across a split, but does not establish that historical series or volume are correctly adjusted. These limits stay attached to any final conditional result. Headerless or conflicting supplements are not silently merged.


In [ ]:
coverage_table = pd.DataFrame(COVERAGE_ROWS)
display(coverage_table)
symbols = CFG['universe'] + [CFG['index_symbol']]
coverage_records = []
with sqlite3.connect(DB_READONLY_URI, uri=True) as con:
    for label, lo, hi in [('start','09:15:00','15:29:00'),('end','09:16:00','15:30:00')]:
        rows = con.execute("""
          SELECT symbol,substr(stamp,1,10),count(*) FROM bars
          WHERE substr(stamp,12) BETWEEN ? AND ? GROUP BY symbol,substr(stamp,1,10)
        """, (lo,hi)).fetchall()
        count_map = {(s,d):v for s,d,v in rows}
        for d in CALENDAR:
            for s in symbols:
                coverage_records.append(dict(timestamp_label=label,date=d.isoformat(),symbol=s,
                                             observed_minutes=count_map.get((s,d.isoformat()),0)))
coverage_daily = pd.DataFrame(coverage_records)
off_calendar = sorted({r['date'] for r in RAW_LABEL_DAY_COUNTS} - {d.isoformat() for d in CALENDAR})
print('Raw dates outside the declared normal-session calendar:', off_calendar)
fig, axes = plt.subplots(2,1,figsize=(12,4.8),sharex=True)
for ax,label in zip(axes,['start','end']):
    grid = coverage_daily.query('timestamp_label == @label').pivot(index='symbol',columns='date',values='observed_minutes').reindex(symbols)
    img = ax.imshow(grid.values,aspect='auto',vmin=0,vmax=375,cmap='viridis')
    ticks = np.linspace(0,len(grid.columns)-1,min(7,len(grid.columns))).astype(int)
    ax.set(yticks=range(len(symbols)),yticklabels=symbols,title=f'{label}-label assumption: normal-session coverage',
           xticks=ticks,xticklabels=grid.columns[ticks])
    fig.colorbar(img,ax=ax,label='Minutes observed')
axes[-1].set_xlabel('Scheduled date'); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'coverage.png'); plt.show()


## 4. Signal and sizing: the quantity available at a decision

For a decision at time t, let the latest completed close be C(t−1). The fifteen-minute stock return is

$$
r_{i,d,t}=\log\frac{C_{i,d,t-1}}{C_{i,d,t-16}},\qquad
x_{i,d,t}=r_{i,d,t}-r_{m,d,t},\qquad
z_{i,d,t}=\frac{x_{i,d,t}-\bar{x}_{i,t}^{(20)}}{s_{x,i,t}^{(20)}}.
$$

Both instruments require every minute in that sixteen-label window. The mean and sample SD use the **same clock time in exactly the preceding twenty scheduled sessions**. Missing history cannot be replaced by older complete sessions. A qualifying stock can have a positive return if the index rose more. The z-score is not a calibrated tail probability.

The barrier scale is different: $v=s_{r,i,t}^{(20)}$, the stock-only return sample SD. After observing the entry open P, the fixed stop and target are $Pe^{-v}$ and $Pe^{1.5v}$. The entry is delayed by the scenario's declared number of minutes; all policies use that same open and expiry. At a one-minute delay, a 09:45 decision enters at 09:46 and expires at 10:01. Each entry notional is INR 10,000. Early exits cannot create extra opportunities or free capital before common expiry.


## Visible research method

The following cells contain the entire signal, execution and accounting method. They do not import project modules or launch hidden scripts. Read only one scheduled session from SQLite at a time. The signal uses stock-minus-index returns; the index coefficient is one and there is no traded hedge. Entry signals and barriers use preceding scheduled sessions only.

The primary output is a conditional complete-case independent-session comparison. The separate funded diagnostic carries cash, values positions at minute closes and reserves early proceeds until common expiry. Known unavailable pre-entry inputs cause logged abstention in the funded diagnostic; unknown post-entry prices stop the identified funded record. Funding failure is terminal. Each predeclared evaluation-period invocation starts at C0; restarting within it requires an explicit separately labelled segment and is forbidden after economic funding failure. Neither compressed minute valuations nor assumed barrier fills recover actual ticks.


### Configuration and one-session SQLite reader


In [ ]:
from collections import Counter, deque
from contextlib import closing
from datetime import date, datetime, time, timedelta, timezone
from math import exp, fsum, isfinite, log
from pathlib import Path
from statistics import mean, stdev
import sqlite3

IST = timezone(timedelta(hours=5, minutes=30))
MINUTE = timedelta(minutes=1)
POLICIES = ("time_only", "stop_only", "stop_and_target")
PATHS = ("O-L-H-C", "O-H-L-C")
SCENARIOS = tuple((policy, path) for path in PATHS for policy in POLICIES)


def calendar_date(value):
    if isinstance(value, datetime):
        return value.date()
    return value if isinstance(value, date) else date.fromisoformat(str(value)[:10])


def research_parameters(cfg):
    """A single declared scenario; this function performs no parameter selection."""
    p = {
        "universe": tuple(cfg["universe"]), "index": cfg.get("index_symbol", ".NSEI"),
        "C0": float(cfg.get("reference_capital_inr", 100000)),
        "allocation": float(cfg.get("allocation_fraction", 0.5)),
        "lookback": int(cfg.get("prior_sessions", 20)),
        "window": int(cfg.get("signal_window_minutes", 15)),
        "threshold": float(cfg.get("z_threshold", -2)),
        "first": int(cfg.get("decision_minutes_after_open", 30)),
        "interval": int(cfg.get("decision_interval_minutes", 30)),
        "holding": int(cfg.get("holding_minutes", 15)),
        "delay": int(cfg.get("delay_minutes", 1)),
        "cost": float(cfg.get("cost_bp", 15)),
        "slippage": float(cfg.get("stop_slippage_bp", 0)),
        "label": cfg.get("timestamp_label", "start"),
        "open": time.fromisoformat(cfg.get("normal_session_open", "09:15")),
        "close": time.fromisoformat(cfg.get("normal_session_close", "15:30")),
    }
    if (not p["universe"] or len(set(p["universe"])) != len(p["universe"])
            or p["index"] in p["universe"]):
        raise ValueError("Declare a distinct stock universe and separate index")
    if p["label"] not in ("start", "end"):
        raise ValueError("timestamp_label must be start or end; neither confirms supplier semantics")
    if not all(isfinite(p[k]) for k in ("C0", "allocation", "threshold", "cost", "slippage")):
        raise ValueError("Parameters must be finite")
    if p["C0"] <= 0 or not 0 < p["allocation"] < 1 or min(p["cost"], p["slippage"]) < 0:
        raise ValueError("Invalid capital, allocation or costs")
    if (p["lookback"] < 2 or min(p["window"], p["delay"], p["holding"]) < 1
            or p["first"] < p["window"] + 1 or p["interval"] < p["holding"]):
        raise ValueError("Invalid history, timing or overlapping scheduled holding windows")
    if p["open"] >= p["close"]:
        raise ValueError("Invalid session bounds")
    p["notional"] = p["allocation"] * p["C0"] / len(p["universe"])
    return p


def session_bounds(day, p):
    return (datetime.combine(day, p["open"], IST),
            datetime.combine(day, p["close"], IST))


def read_session(connection, day, p):
    """Read only one day/symbol at a time. SQLite stores unchanged naive source labels."""
    start, end = session_bounds(day, p)
    output = {}
    for symbol in (*p["universe"], p["index"]):
        values = {}
        rows = connection.execute(
            "SELECT stamp,o,h,l,c FROM bars WHERE symbol=? AND stamp>=? AND stamp<? ORDER BY stamp",
            (symbol, day.isoformat() + "T00:00:00", (day + timedelta(days=1)).isoformat() + "T00:00:00"))
        for raw, opening, high, low, close in rows:
            stamp = datetime.fromisoformat(raw)
            if stamp.tzinfo is not None or stamp.second or stamp.microsecond:
                raise ValueError("Expected naive, exact-minute source labels")
            stamp = stamp.replace(tzinfo=IST) - (MINUTE if p["label"] == "end" else timedelta(0))
            prices = tuple(float(x) for x in (opening, high, low, close))
            o, h, l, c = prices
            if not all(isfinite(x) and x > 0 for x in prices) or not l <= min(o, c) <= max(o, c) <= h:
                raise ValueError(f"Invalid OHLC: {symbol} {raw}")
            if stamp in values:
                raise ValueError(f"Duplicate transformed label: {symbol} {stamp}")
            if start <= stamp < end:
                values[stamp] = {"o": o, "h": h, "l": l, "c": c}
        output[symbol] = values
    return output

### Past-only signal and common episode schedule


In [ ]:
def scheduled_decisions(day, p):
    opening, closing = session_bounds(day, p)
    at = opening + p["first"] * MINUTE
    decisions = []
    while at + (p["delay"] + p["holding"]) * MINUTE < closing:
        decisions.append(at)
        at += p["interval"] * MINUTE
    return decisions


def window_feature(stock, index, decision, window):
    labels = [decision - k * MINUTE for k in range(window + 1, 0, -1)]
    if any(at not in stock or at not in index for at in labels):
        return None
    rs = log(stock[labels[-1]]["c"] / stock[labels[0]]["c"])
    ri = log(index[labels[-1]]["c"] / index[labels[0]]["c"])
    return (rs, rs - ri)


def plan_session(day, bars, history, p):
    """History contains exactly preceding scheduled sessions, including missing maps."""
    features, candidates, episodes = {}, [], []
    index = bars.get(p["index"], {})
    for decision in scheduled_decisions(day, p):
        for symbol in p["universe"]:
            key = (symbol, decision.time())
            value = window_feature(bars.get(symbol, {}), index, decision, p["window"])
            features[key] = value
            row = {"date": day.isoformat(), "symbol": symbol, "decision_at": decision.isoformat(),
                   "status": "MISSING_CURRENT_WINDOW", "z": None, "stock_volatility": None}
            if value is not None:
                row.update(stock_return=value[0], relative_return=value[1])
                if len(history) < p["lookback"]:
                    row["status"] = "WARMUP"
                else:
                    prior = [entry[1].get(key) for entry in history]
                    if any(x is None for x in prior):
                        row["status"] = "MISSING_HISTORY_WINDOW"
                    else:
                        relative_mean = mean(x[1] for x in prior)
                        relative_sd = stdev(x[1] for x in prior)
                        stock_sd = stdev(x[0] for x in prior)
                        row.update(prior_relative_mean=relative_mean, prior_relative_sd=relative_sd,
                                   stock_volatility=stock_sd)
                        if relative_sd == 0 or stock_sd == 0:
                            row["status"] = "UNDEFINED_SCALE"
                        else:
                            row["z"] = (value[1] - relative_mean) / relative_sd
                            row["status"] = "SIGNAL" if row["z"] <= p["threshold"] else "NO_SIGNAL"
            candidates.append(row)
            if row["status"] == "SIGNAL":
                entry = decision + p["delay"] * MINUTE
                episodes.append({"episode_id": f"{day}_{symbol}_{decision:%H%M}", "symbol": symbol,
                                 "signal_at": decision, "entry_at": entry,
                                 "expiry_at": entry + p["holding"] * MINUTE,
                                 "stop_multiplier": exp(-row["stock_volatility"]),
                                 "target_multiplier": exp(1.5 * row["stock_volatility"]),
                                 "notional": p["notional"]})
    return features, candidates, episodes

### Visible execution and self-financing cash/position accounting


In [ ]:
def fresh_accounts(capital):
    return {key: {"cash": float(capital), "positions": {}, "reserved": {}, "peak": float(capital),
                  "last_equity": None, "last_cash": None, "last_reserved": None}
            for key in SCENARIOS}


def barrier_fill(position, bar, at, policy, path):
    """At expiry only its opening price matters. No later high/low can precede that exit."""
    if at == position["expiry_at"]:
        return "TIME", bar["o"], "open", False
    if policy == "time_only":
        return None
    stop, target = position["stop"], position["target"]
    if bar["o"] <= stop:
        return "STOP", bar["o"], "open", False
    if policy == "stop_and_target" and bar["o"] >= target:
        return "TARGET", target, "open", False
    stop_hit = bar["l"] <= stop
    target_hit = policy == "stop_and_target" and bar["h"] >= target
    if stop_hit and target_hit:
        return ("STOP", stop, "within_bar", True) if path == "O-L-H-C" else ("TARGET", target, "within_bar", True)
    if stop_hit:
        return "STOP", stop, "within_bar", False
    if target_hit:
        return "TARGET", target, "within_bar", False
    return None


def release_cash(account, at):
    released = []
    for episode_id, (expiry, amount) in list(account["reserved"].items()):
        if expiry <= at:
            del account["reserved"][episode_id]
            released.append((episode_id, amount))
    return released


def available_cash(account):
    return account["cash"] - fsum(amount for _, amount in account["reserved"].values())


def close_position(account, position, at, fill, p, policy, path, accounting, segment):
    reason, raw_price, phase, ambiguous = fill
    extra = p["slippage"] if reason == "STOP" else 0.0
    exit_price = raw_price - position["entry_price"] * extra / 10000
    if exit_price <= 0:
        raise ValueError("Declared adverse slippage produces a nonpositive exit price")
    exit_fee = position["notional"] * p["cost"] / 20000
    proceeds = position["shares"] * exit_price - exit_fee
    account["cash"] += proceeds
    if at < position["expiry_at"] and proceeds > 0:
        account["reserved"][position["episode_id"]] = (position["expiry_at"], proceeds)
    del account["positions"][position["symbol"]]
    net_pnl = proceeds - position["notional"] - position["entry_fee"]
    held = (at - position["entry_at"]).total_seconds() / 60
    return {"date": at.date().isoformat(), "accounting": accounting, "segment": segment,
            "policy": policy, "path": path, "episode_id": position["episode_id"],
            "symbol": position["symbol"], "status": "RESOLVED",
            "signal_at": position["signal_at"].isoformat(), "entry_at": position["entry_at"].isoformat(),
            "expiry_at": position["expiry_at"].isoformat(), "exit_at": at.isoformat(), "exit_phase": phase,
            "entry_price": position["entry_price"], "stop_price": position["stop"], "target_price": position["target"],
            "exit_price": exit_price, "exit_reason": reason, "notional": position["notional"],
            "entry_fee": position["entry_fee"], "exit_fee": exit_fee,
            "extra_stop_slippage_bp": extra, "net_pnl": net_pnl,
            "net_return_bp": net_pnl / position["notional"] * 10000,
            "ambiguous": ambiguous, "holding_lower_minutes": held,
            "holding_upper_minutes": held + (1 if phase == "within_bar" else 0)}


def replay_session(day, bars, episodes, accounts, p, accounting, segment, equity_sink=None):
    """Six matched accounts; no hidden fills, borrowing, capital recycling or automatic resets.

    An unknown required bar ends the common identified record BEFORE that minute's events.
    Flat periods need no market price. Every minute close is valued; identical consecutive
    valuations are compressed. Therefore drawdown is unchanged, but this is not tick risk.
    """
    opening, closing = session_bounds(day, p)
    start_cash = {key: a["cash"] for key, a in accounts.items()}
    if any(a["positions"] or a["reserved"] for a in accounts.values()):
        raise ValueError("A new session requires flat, identified accounts with released cash")
    by_entry = {}
    for ep in episodes:
        if (not opening <= ep["entry_at"] < ep["expiry_at"] < closing
                or ep["entry_at"] - ep["signal_at"] < MINUTE):
            raise ValueError("Episode timing is outside the declared intraday schedule")
        by_entry.setdefault(ep["entry_at"], []).append(ep)
    previous_expiry = None
    for at, batch in sorted(by_entry.items()):
        if (len({ep["symbol"] for ep in batch}) != len(batch)
                or len({ep["expiry_at"] for ep in batch}) != 1
                or (previous_expiry is not None and at < previous_expiry)):
            raise ValueError("Common episode batches must be nonoverlapping and have one entry per stock")
        previous_expiry = batch[0]["expiry_at"]
    trades, equity, cash_events, failure = [], [], [], None
    minimum_cash = {key: available_cash(a) for key, a in accounts.items()}

    def execute_exit(key, account, position, at, fill):
        trade = close_position(account, position, at, fill, p, *key, accounting, segment)
        trades.append(trade)
        cash_events.append({"date": day.isoformat(), "timestamp": at.isoformat(),
                            "accounting": accounting, "segment": segment, "policy": key[0], "path": key[1],
                            "event": "EXIT_" + trade["exit_reason"], "episode_id": trade["episode_id"],
                            "phase": trade["exit_phase"],
                            "cash_delta": position["shares"] * trade["exit_price"] - trade["exit_fee"],
                            "available_cash": available_cash(account)})

    def emit_equity(at, kind, force=False):
        for (policy, path), account in accounts.items():
            market_value = fsum(pos["shares"] * bars[symbol][at - MINUTE]["c"]
                                for symbol, pos in account["positions"].items()) if kind == "minute_close" else 0.0
            value = account["cash"] + market_value
            reserved = fsum(x[1] for x in account["reserved"].values())
            account["peak"] = max(account["peak"], value)
            changed = (value, account["cash"], reserved) != (account["last_equity"], account["last_cash"], account["last_reserved"])
            if force or changed:
                row = {"date": day.isoformat(), "timestamp": at.isoformat(), "accounting": accounting,
                       "segment": segment, "policy": policy, "path": path, "sampling": kind,
                       "cash": account["cash"], "available_cash": available_cash(account),
                       "reserved_cash": reserved, "market_value": market_value, "equity": value,
                       "drawdown_fraction": 1 - value / account["peak"],
                       "open_positions": len(account["positions"])}
                if equity_sink is None:
                    equity.append(row)
                else:
                    equity_sink(row)
            account["last_equity"], account["last_cash"], account["last_reserved"] = value, account["cash"], reserved

    if accounting == "funded":
        emit_equity(opening, "session_open", True)
    at = opening
    while at < closing:
        new = by_entry.get(at, [])
        required = {ep["symbol"] for ep in new}
        required.update(symbol for a in accounts.values() for symbol in a["positions"])
        absent = sorted(symbol for symbol in required if at not in bars.get(symbol, {}))
        if absent:
            failure = {"status": "UNKNOWN_EQUITY", "at": at.isoformat(), "symbols": absent,
                       "reason": "Missing required entry, exit or held-position valuation bar"}
            for key in accounts:
                for ep in new:
                    trades.append({"date": day.isoformat(), "accounting": accounting, "segment": segment,
                                   "policy": key[0], "path": key[1], "episode_id": ep["episode_id"],
                                   "symbol": ep["symbol"], "status": "UNRESOLVED_ENTRY", "net_pnl": None,
                                   "unresolved_at": at.isoformat(), "reason": failure["reason"]})
            break
        for key, account in accounts.items():
            released = release_cash(account, at)
            if released:
                cash_events.append({"date": day.isoformat(), "timestamp": at.isoformat(),
                                    "accounting": accounting, "segment": segment, "policy": key[0], "path": key[1],
                                    "event": "COMMON_EXPIRY_RELEASE", "cash_delta": 0.0,
                                    "released_cash": fsum(amount for _, amount in released),
                                    "available_cash": available_cash(account)})
            # Earlier expiry exits precede any new entries and all later intraminute extremes.
            for position in list(account["positions"].values()):
                if at == position["expiry_at"]:
                    execute_exit(key, account, position, at,
                                 ("TIME", bars[position["symbol"]][at]["o"], "open", False))
        if new:
            required_cash = fsum(ep["notional"] * (1 + p["cost"] / 20000) for ep in new)
            shortages = [{"policy": key[0], "path": key[1], "available_cash": available_cash(a),
                          "required_cash": required_cash, "known_equity": a["cash"]}
                         for key, a in accounts.items() if available_cash(a) + 1e-8 < required_cash]
            if shortages:
                failure = {"status": "FUNDING_FAILURE", "at": at.isoformat(), "accounts": shortages,
                           "reason": "A matched account cannot fund all common entries including fees"}
                if accounting == "funded":
                    # Adjacent batches may have closed prior positions at this open.
                    # Record that known final cash instead of leaving a stale last mark.
                    if any(a["positions"] for a in accounts.values()):
                        raise AssertionError("Funding failure boundary must be flat after common expiry")
                    emit_equity(at, "funding_failure_open", True)
                break
            for key, account in accounts.items():
                for ep in new:
                    if ep["symbol"] in account["positions"]:
                        raise ValueError("Overlapping same-stock entry or forbidden early re-entry")
                    price = bars[ep["symbol"]][at]["o"]
                    fee = ep["notional"] * p["cost"] / 20000
                    account["cash"] -= ep["notional"] + fee
                    account["positions"][ep["symbol"]] = dict(ep, entry_price=price,
                        shares=ep["notional"] / price, entry_fee=fee,
                        stop=price * ep["stop_multiplier"], target=price * ep["target_multiplier"])
                cash_events.append({"date": day.isoformat(), "timestamp": at.isoformat(),
                                    "accounting": accounting, "segment": segment, "policy": key[0], "path": key[1],
                                    "event": "COMMON_ENTRY", "required_cash": required_cash,
                                    "cash_delta": -required_cash,
                                    "available_cash": available_cash(account)})
        exit_cost_shortages = []
        for key, account in accounts.items():
            minimum_cash[key] = min(minimum_cash[key], available_cash(account))
            for position in list(account["positions"].values()):
                fill = barrier_fill(position, bars[position["symbol"]][at], at, *key)
                if fill is not None:
                    execute_exit(key, account, position, at, fill)
            minimum_cash[key] = min(minimum_cash[key], available_cash(account))
            if available_cash(account) < -1e-7 or account["cash"] < -1e-7:
                known_value = account["cash"] + fsum(pos["shares"] * bars[symbol][at]["c"]
                                                     for symbol, pos in account["positions"].items())
                exit_cost_shortages.append({"policy": key[0], "path": key[1],
                                            "available_cash": available_cash(account), "cash": account["cash"],
                                            "additional_funding_required": max(0.0, -available_cash(account)),
                                            "known_equity_at_bar_close": known_value})
        if exit_cost_shortages:
            failure = {"status": "FUNDING_FAILURE", "at": at.isoformat(),
                       "accounts": exit_cost_shortages, "phase": "bar_outcome",
                       "reason": "Realised exit costs create a cash deficit; terminal economic failure, no continued borrowing"}
            if accounting == "funded":
                emit_equity(at + MINUTE, "minute_close", True)
            break
        if accounting == "funded":
            emit_equity(at + MINUTE, "minute_close", force=(at + MINUTE == closing))
        at += MINUTE
    if failure is not None:
        for key, account in accounts.items():
            for pos in account["positions"].values():
                trades.append({"date": day.isoformat(), "accounting": accounting, "segment": segment,
                               "policy": key[0], "path": key[1], "episode_id": pos["episode_id"],
                               "symbol": pos["symbol"],
                               "status": "UNRESOLVED" if failure["status"] == "UNKNOWN_EQUITY" else "OPEN_AT_FUNDING_FAILURE",
                               "net_pnl": None,
                               "unresolved_at": failure["at"], "reason": failure["reason"]})
    else:
        for account in accounts.values():
            release_cash(account, closing)
        if any(a["positions"] or a["reserved"] for a in accounts.values()):
            raise AssertionError("Normal session ended with an open position or reserved proceeds")
    daily = []
    for key, account in accounts.items():
        pnl = None if failure else account["cash"] - start_cash[key]
        ledger_pnl = fsum(r["net_pnl"] for r in trades if (r["policy"], r["path"]) == key and r["status"] == "RESOLVED")
        cash_change = fsum(r["cash_delta"] for r in cash_events if (r["policy"], r["path"]) == key)
        if abs(account["cash"] - start_cash[key] - cash_change) > max(1e-7, abs(account["cash"]) * 1e-12):
            raise AssertionError("Cash flow ledger does not reconcile to physical cash")
        if failure is None and abs(pnl - ledger_pnl) > max(1e-7, abs(account["cash"]) * 1e-12):
            raise AssertionError("Cash change does not reconcile to realised trade P&L")
        daily.append({"date": day.isoformat(), "accounting": accounting, "segment": segment,
                      "policy": key[0], "path": key[1], "status": failure["status"] if failure else "EVALUABLE",
                      "net_pnl": pnl, "return_on_reference_capital": None if pnl is None else pnl / p["C0"],
                      "opening_cash": start_cash[key], "closing_cash": None if failure else account["cash"],
                      "minimum_available_cash": minimum_cash[key],
                      "unique_planned_episodes": len(episodes), "failure": failure})
    return {"daily": daily, "trades": trades, "equity": equity, "cash_events": cash_events, "failure": failure}

### End-to-end replay: calendar history, conditional means, and funded segments


In [ ]:
def run_research(DB_PATH, CFG, CALENDAR, evaluation_dates, *, equity_sink=None):
    """No market outcomes are read outside the supplied calendar slice.

    evaluation_dates must be a predeclared contiguous calendar slice, not selected complete days.
    Use a fresh invocation for validation and final test: each funded diagnostic starts at C0.
    The only allowed later new-C0 restart is explicit CFG['funded_restart_dates']={date: reason}
    after unknown data. It is a NEW segment; an economic funding failure is terminal.
    """
    p = research_parameters(CFG)
    calendar = [calendar_date(x) for x in CALENDAR]
    if calendar != sorted(set(calendar)):
        raise ValueError("CALENDAR must be unique and strictly increasing")
    evaluate = {calendar_date(x) for x in evaluation_dates}
    if not evaluate or not evaluate <= set(calendar):
        raise ValueError("evaluation_dates must be a nonempty subset of the scheduled calendar")
    selected = [day for day in calendar if min(evaluate) <= day <= max(evaluate)]
    if set(selected) != evaluate:
        raise ValueError("Use a contiguous calendar slice; do not select dates by future completeness")
    calendar = [day for day in calendar if day <= max(evaluate)]
    excluded = {calendar_date(x) for x in CFG.get("excluded_nonstandard_sessions", ())}
    restarts = {calendar_date(k): v for k, v in CFG.get("funded_restart_dates", {}).items()}
    if not set(restarts) <= evaluate or any(not str(v).strip() for v in restarts.values()):
        raise ValueError("Every explicit restart needs an evaluation date and reason")
    history = deque(maxlen=p["lookback"])
    output = {name: [] for name in ("daily", "trades", "equity", "cash_events", "diagnostics", "candidates", "segments")}
    funded, funded_status, segment = None, "NOT_STARTED", 0
    uri = Path(DB_PATH).resolve().as_uri() + "?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as connection:
        connection.execute("PRAGMA cache_size=-2048")
        connection.execute("PRAGMA temp_store=FILE")
        for day in calendar:
            bars = {} if day in excluded else read_session(connection, day, p)
            features, candidates, episodes = plan_session(day, bars, history, p)
            if day in evaluate:
                if funded is None:
                    segment += 1
                    funded = fresh_accounts(p["C0"])
                    funded_status = "ACTIVE"
                    output["segments"].append({"segment": segment, "start_date": day.isoformat(),
                                               "initial_capital": p["C0"], "reason": "Predeclared evaluation boundary"})
                elif day in restarts:
                    if funded_status != "UNKNOWN_EQUITY":
                        raise ValueError("A restart is allowed only after unknown data, never a funding failure or active segment")
                    segment += 1
                    funded = fresh_accounts(p["C0"])
                    funded_status = "ACTIVE"
                    output["segments"].append({"segment": segment, "start_date": day.isoformat(),
                                               "initial_capital": p["C0"], "reason": str(restarts[day])})
                counts = Counter(row["status"] for row in candidates)
                invalid = [row for row in candidates if row["status"] not in ("SIGNAL", "NO_SIGNAL")]
                primary_status = ("EXCLUDED_SESSION" if day in excluded else
                                  "INCOMPLETE_CANDIDATE_COVERAGE" if invalid or not candidates else "EVALUABLE")
                if day in excluded:
                    episodes = []
                independent = replay_session(day, bars, episodes, fresh_accounts(p["C0"]), p,
                                             "independent", 0)
                if independent["failure"] is not None:
                    primary_status = independent["failure"]["status"]
                if primary_status != "EVALUABLE":
                    for row in independent["daily"]:
                        row.update(status=primary_status, net_pnl=None, return_on_reference_capital=None, closing_cash=None)
                for row in independent["trades"]:
                    row["included_primary_day"] = primary_status == "EVALUABLE"
                for name in ("daily", "trades", "cash_events"):
                    output[name].extend(independent[name])
                if funded_status == "ACTIVE":
                    # Known unavailable pre-entry inputs create abstentions. No unknown prices are imputed.
                    current = replay_session(day, bars, episodes, funded, p, "funded", segment, equity_sink)
                    for name in ("daily", "trades", "equity", "cash_events"):
                        output[name].extend(current[name])
                    if current["failure"] is not None:
                        funded_status = current["failure"]["status"]
                        output["segments"][-1].update(end_date=day.isoformat(), end_reason=funded_status,
                                                      failure=current["failure"])
                else:
                    for policy, path in SCENARIOS:
                        output["daily"].append({"date": day.isoformat(), "accounting": "funded", "segment": segment,
                                                "policy": policy, "path": path, "status": "BLOCKED_" + funded_status,
                                                "net_pnl": None, "return_on_reference_capital": None,
                                                "opening_cash": None, "closing_cash": None})
                output["diagnostics"].append({"date": day.isoformat(), "candidate_counts": dict(counts),
                                              "known_abstentions": len(invalid), "planned_episodes": len(episodes),
                                              "primary_status": primary_status, "funded_status": funded_status,
                                              "segment": segment, "deliberately_excluded": day in excluded})
                if CFG.get("store_candidates", False):
                    output["candidates"].extend(candidates)
                else:
                    output["candidates"].extend(row for row in candidates if row["status"] == "SIGNAL")
            # Appended AFTER today's signals: no current or future observations enter their own history.
            history.append((day, features))
    if output["segments"] and "end_date" not in output["segments"][-1]:
        output["segments"][-1].update(end_date=max(evaluate).isoformat(), end_reason="END_OF_EVALUATION")
    output["metadata"] = {"title": "Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities",
                          "timestamp_label_assumption": p["label"], "timezone_assumption": "Asia/Kolkata",
                          "C0": p["C0"], "cost_bp": p["cost"], "stop_slippage_bp": p["slippage"],
                          "delay_minutes": p["delay"], "evaluation_start": min(evaluate).isoformat(),
                          "evaluation_end": max(evaluate).isoformat(), "terminal_funded_status": funded_status,
                          "equity_sampling": "Minute closes plus known flat opening/failure boundaries; unchanged consecutive values omitted",
                          "primary_estimand": "Complete-case independent-session P&L difference / C0",
                          "funded_estimand": "Separate self-financing fixed-notional feasibility diagnostic",
                          "notes": ["No target-only policy; no parameter search; no automatic capital reset.",
                                    "Explicit excluded sessions are retrospective sample restrictions, not outage-loss estimates.",
                                    "Intraminute fills are assumed and their exact execution seconds are unknown."]}
    return output

### Small synthetic checks of the visible notebook method


In [ ]:
def run_notebook_method_checks():
    """Synthetic checks exercise semantics. They supply no market evidence or holdout result."""
    p = research_parameters({"universe": ["STOCK"], "reference_capital_inr": 1000, "cost_bp": 0})
    day = date(2021, 1, 4)
    start = datetime.combine(day, time(9, 46), IST)
    ep = {"episode_id": "synthetic", "symbol": "STOCK", "signal_at": start - MINUTE,
          "entry_at": start, "expiry_at": start + 15 * MINUTE,
          "stop_multiplier": 0.99, "target_multiplier": 1.015, "notional": 500.0}
    flat = {start + i * MINUTE: {"o": 100.0, "h": 100.0, "l": 100.0, "c": 100.0} for i in range(16)}
    both = dict(flat)
    both[start] = {"o": 100.0, "h": 102.0, "l": 98.0, "c": 100.0}
    accounts = fresh_accounts(1000)
    result = replay_session(day, {"STOCK": both}, [ep], accounts, p, "funded", 1)
    found = {(r["policy"], r["path"]): r for r in result["trades"]}
    assert found[("stop_and_target", "O-L-H-C")]["exit_reason"] == "STOP"
    assert found[("stop_and_target", "O-H-L-C")]["exit_reason"] == "TARGET"
    assert found[("time_only", "O-L-H-C")]["exit_reason"] == "TIME"
    assert found[("stop_and_target", "O-L-H-C")]["ambiguous"]
    before_expiry = [r for r in result["equity"] if r["policy"] == "stop_and_target" and r["path"] == "O-H-L-C"
                     and r["timestamp"] == (start + MINUTE).isoformat()][0]
    assert abs(before_expiry["available_cash"] - 500) < 1e-9
    assert abs(before_expiry["reserved_cash"] - 507.5) < 1e-9
    assert abs(before_expiry["equity"] - 1007.5) < 1e-9
    assert not accounts[("stop_and_target", "O-H-L-C")]["reserved"]
    # The time-only position marks its changing close; no cash-only fake drawdown.
    marked = dict(flat)
    marked[start] = {"o": 100.0, "h": 100.0, "l": 98.0, "c": 98.0}
    marked_result = replay_session(day, {"STOCK": marked}, [ep], fresh_accounts(1000), p, "funded", 1)
    first_mark = next(r for r in marked_result["equity"] if r["policy"] == "time_only" and r["path"] == "O-L-H-C"
                      and r["timestamp"] == (start + MINUTE).isoformat())
    assert abs(first_mark["equity"] - 990) < 1e-9 and abs(first_mark["drawdown_fraction"] - 0.01) < 1e-9
    # Expiry open wins even if later extrema touch both barriers.
    expiry = dict(flat)
    expiry[ep["expiry_at"]] = {"o": 100.25, "h": 110.0, "l": 90.0, "c": 100.0}
    expired = replay_session(day, {"STOCK": expiry}, [ep], fresh_accounts(1000), p, "independent", 0)
    assert all(r["exit_reason"] == "TIME" and r["exit_price"] == 100.25 for r in expired["trades"])
    # Opening gap at a stop, with slippage debited once.
    gap = dict(flat)
    gap[start + MINUTE] = {"o": 98.0, "h": 98.0, "l": 98.0, "c": 98.0}
    slip_p = dict(p, slippage=5.0)
    gapped = replay_session(day, {"STOCK": gap}, [ep], fresh_accounts(1000), slip_p, "independent", 0)
    stopped = next(r for r in gapped["trades"] if r["policy"] == "stop_only")
    assert abs(stopped["exit_price"] - 97.95) < 1e-9 and abs(stopped["net_pnl"] + 10.25) < 1e-9
    # Missing held-position bar halts common equity; it is not a zero or interpolated price.
    missing = dict(flat)
    del missing[start + MINUTE]
    unresolved = replay_session(day, {"STOCK": missing}, [ep], fresh_accounts(1000), p, "funded", 1)
    assert unresolved["failure"]["status"] == "UNKNOWN_EQUITY"
    assert all(r["net_pnl"] is None for r in unresolved["daily"])
    # A funding shortage in ONE account prevents entries in ALL matched accounts.
    short = fresh_accounts(1000)
    short[("stop_only", "O-L-H-C")]["cash"] = 499.0
    failed = replay_session(day, {"STOCK": flat}, [ep], short, p, "funded", 1)
    assert failed["failure"]["status"] == "FUNDING_FAILURE" and not failed["trades"]
    assert all(not a["positions"] for a in short.values())
    # An adjacent expiry-open gain must be marked before a new batch fails funding.
    adjacent_bars = dict(flat)
    adjacent_bars[ep["expiry_at"]] = {"o": 100.25, "h": 100.25, "l": 100.25, "c": 100.25}
    adjacent_ep = dict(ep, episode_id="adjacent", signal_at=ep["expiry_at"] - MINUTE,
                       entry_at=ep["expiry_at"], expiry_at=ep["expiry_at"] + 15 * MINUTE, notional=1002.0)
    adjacent = replay_session(day, {"STOCK": adjacent_bars}, [ep, adjacent_ep], fresh_accounts(1000), p, "funded", 1)
    final_mark = next(r for r in reversed(adjacent["equity"]) if r["policy"] == "time_only" and r["path"] == "O-L-H-C")
    assert adjacent["failure"]["status"] == "FUNDING_FAILURE"
    assert final_mark["sampling"] == "funding_failure_open" and abs(final_mark["equity"] - 1001.25) < 1e-9
    # A realised exit-fee deficit is an economic failure, not an assertion or hidden loan.
    crashed_bars = dict(flat)
    crashed_bars[start + MINUTE] = {"o": 0.001, "h": 0.001, "l": 0.001, "c": 0.001}
    thin_cash = fresh_accounts(1000)
    for account in thin_cash.values():
        account["cash"] = 500.375  # Exactly covers INR500 entry and half of 15 bp cost.
    crashed = replay_session(day, {"STOCK": crashed_bars}, [ep], thin_cash, dict(p, cost=15.0), "funded", 1)
    assert crashed["failure"]["status"] == "FUNDING_FAILURE"
    assert "cash deficit" in crashed["failure"]["reason"]
    cash_deficit = next(a for a in crashed["failure"]["accounts"] if a["policy"] == "stop_only")
    assert abs(cash_deficit["cash"] + 0.37) < 1e-9
    assert abs(cash_deficit["additional_funding_required"] - 0.37) < 1e-9
    assert any(r["status"] == "RESOLVED" and r["exit_reason"] == "STOP" for r in crashed["trades"])
    assert any(r["status"] == "OPEN_AT_FUNDING_FAILURE" for r in crashed["trades"])
    # A second day carries cash, rather than resetting prior profits to C0.
    next_day = day + timedelta(days=1)
    next_bars = {t + timedelta(days=1): b for t, b in flat.items()}
    next_ep = dict(ep, episode_id="next", signal_at=ep["signal_at"] + timedelta(days=1),
                   entry_at=ep["entry_at"] + timedelta(days=1), expiry_at=ep["expiry_at"] + timedelta(days=1))
    carried = replay_session(next_day, {"STOCK": next_bars}, [next_ep], accounts, p, "funded", 1)
    carry_row = next(r for r in carried["daily"] if r["policy"] == "stop_and_target" and r["path"] == "O-H-L-C")
    assert abs(carry_row["opening_cash"] - 1007.5) < 1e-9
    # A close-to-close feature uses only the sixteen completed labels.
    decision = start - MINUTE
    labels = [decision - k * MINUTE for k in range(16, 0, -1)]
    stock = {t: {"c": 100 + i / 10} for i, t in enumerate(labels)}
    index = {t: {"c": 100} for t in labels}
    original = window_feature(stock, index, decision, 15)
    stock[decision + 10 * MINUTE] = {"c": 1e12}
    assert window_feature(stock, index, decision, 15) == original
    del stock[labels[5]]
    assert window_feature(stock, index, decision, 15) is None
    # A missing prior session stays missing; an earlier valid date cannot replace it.
    hp = dict(p, lookback=2)
    features = {(symbol, t.time()): (0.001, 0.001)
                for t in scheduled_decisions(day, hp) for symbol in hp["universe"]}
    historical = deque([(day - timedelta(days=2), features), (day - timedelta(days=1), {})], maxlen=2)
    current = {"STOCK": {t: {"c": 100} for t in labels}, hp["index"]: index}
    _, candidate_rows, _ = plan_session(day, current, historical, hp)
    assert candidate_rows[0]["status"] == "MISSING_HISTORY_WINDOW"
    # End-to-end lifecycle on a small synthetic SQLite database, not instructor data.
    import tempfile
    with tempfile.TemporaryDirectory() as temporary:
        database = Path(temporary) / "synthetic.sqlite"
        dates = [date(2021, 1, 4) + timedelta(days=i) for i in range(5)]
        with closing(sqlite3.connect(database)) as connection:
            connection.execute("CREATE TABLE bars(symbol TEXT,stamp TEXT,o REAL,h REAL,l REAL,c REAL,volume REAL,source_id INTEGER,PRIMARY KEY(symbol,stamp)) WITHOUT ROWID")
            for synthetic_day, slope in zip(dates, (0.001, -0.001, -0.01, 0.001, -0.001)):
                opening = datetime.combine(synthetic_day, time(9, 15))
                for i in range(375):
                    stamp = opening + i * MINUTE
                    for symbol, rate in (("STOCK", slope), (".NSEI", 0.0)):
                        o = 100 * exp(rate * i / 15)
                        c = 100 * exp(rate * (i + 1) / 15)
                        connection.execute("INSERT INTO bars VALUES(?,?,?,?,?,?,?,?)",
                                           (symbol, stamp.isoformat(), o, max(o, c), min(o, c), c, 1, 1))
            connection.commit()
        cfg = {"universe": ["STOCK"], "reference_capital_inr": 1000, "prior_sessions": 2,
               "cost_bp": 20, "allocation_fraction": 0.99}
        funded_failure = run_research(database, cfg, dates, dates[2:])
        assert funded_failure["metadata"]["terminal_funded_status"] == "FUNDING_FAILURE"
        assert any(r["status"] == "BLOCKED_FUNDING_FAILURE" for r in funded_failure["daily"])
        assert len(funded_failure["segments"]) == 1
        with closing(sqlite3.connect(database)) as connection:
            connection.execute("DELETE FROM bars WHERE symbol='STOCK' AND stamp=?", (dates[2].isoformat() + "T09:47:00",))
            connection.commit()
        cfg.update(allocation_fraction=0.5, cost_bp=0)
        unknown = run_research(database, cfg, dates, dates[2:])
        assert unknown["metadata"]["terminal_funded_status"] == "UNKNOWN_EQUITY"
        assert len(unknown["segments"]) == 1
        assert any(r["status"] == "BLOCKED_UNKNOWN_EQUITY" for r in unknown["daily"])
        restarted = run_research(database, dict(cfg, funded_restart_dates={dates[4].isoformat(): "Synthetic independent segment"}), dates, dates[2:])
        assert len(restarted["segments"]) == 2
        restart_rows = [r for r in restarted["daily"] if r["accounting"] == "funded" and r["segment"] == 2]
        assert all(r["opening_cash"] == 1000 for r in restart_rows)
        assert restarted["segments"][1]["reason"] == "Synthetic independent segment"
        # Normalisation changes labels only; no actual supplier convention is inferred.
        with closing(sqlite3.connect(database)) as connection:
            raw_start = read_session(connection, dates[0], research_parameters(cfg))
            raw_end = read_session(connection, dates[0], research_parameters(dict(cfg, timestamp_label="end")))
        first = datetime.combine(dates[0], time(9, 15), IST)
        assert raw_end["STOCK"][first] == raw_start["STOCK"][first + MINUTE]
    return {"synthetic_scenarios": 17, "status": "passed", "market_evidence": False}

METHOD_CHECKS = run_notebook_method_checks()
METHOD_CHECKS

### A worked signal and an intentionally ambiguous candle

These examples are **synthetic arithmetic checks**, not NSE observations or evidence of trading profits. The first shows each term of the signal calculation; the second holds an entry open at 100, stop at 99 and target at 101 while a candle reaches both 98 and 102. Only the assumed ordering changes. The two paths are illustrations, not inferred price trajectories.


In [ ]:
past_stock = np.array([.001*np.sin(i)+.0002*i for i in range(20)])
past_index = np.array([.0005*np.cos(i) for i in range(20)])
past_relative = past_stock-past_index
current_stock, current_index = -.009, -.001
worked = dict(stock_return=current_stock,index_return=current_index,index_adjusted_return=current_stock-current_index,
              prior_relative_mean=past_relative.mean(),prior_relative_sample_sd=past_relative.std(ddof=1),
              stock_barrier_scale=past_stock.std(ddof=1))
worked['z'] = (worked['index_adjusted_return']-worked['prior_relative_mean'])/worked['prior_relative_sample_sd']
worked['stop_at_entry_100'] = 100*np.exp(-worked['stock_barrier_scale'])
worked['target_at_entry_100'] = 100*np.exp(1.5*worked['stock_barrier_scale'])
display(pd.DataFrame(worked.items(),columns=['synthetic quantity','value']))
assert worked['z'] < -2 and np.isfinite(worked['z'])
fig, ax = plt.subplots(figsize=(7,3))
for values,label in [([100,98,102,100],'O-L-H-C: stop first'),([100,102,98,100],'O-H-L-C: target first')]:
    ax.plot(range(4),values,'o-',label=label)
ax.axhline(99,color='crimson',ls='--',label='Stop 99'); ax.axhline(101,color='seagreen',ls='--',label='Target 101')
ax.set(xticks=range(4),xticklabels=['Open','First extreme','Second extreme','Close'],
       ylabel='Synthetic price units',xlabel='Assumed ordering (not elapsed seconds)',title='Synthetic both-hit candle: identical OHLC, different exit')
ax.legend(fontsize=8,ncol=2); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'synthetic_ambiguity.png'); plt.show()


## 5. Uncertainty and tail-risk functions, specified before results

Let $R_d^p=\mathrm{P\&L}_d^p/C_0$. The primary point estimate is
$\widehat\Delta=|D|^{-1}\sum_{d\in D}(R_d^{combined}-R_d^{time})$,
where D is the common complete-case day set, including identified zero-trade days. Trade copies across policies and scenarios do not increase the number of independent days.

The implemented interval resamples **whole paired daily vectors** in fixed-length moving blocks, separately within contiguous eligible calendar segments. Every replication preserves each segment's original length: draw eligible blocks uniformly with replacement, concatenate them and truncate only the final sampled block. Blocks never cross unavailable dates. No original day or residual segment is dropped from the point estimate. If any original segment is shorter than the chosen block length, or the sum of floor(segment length / block length) is below ten, the interval is withheld. The threshold is a reporting rule, not a theorem about effective sample size.

This gap-respecting adaptation draws on fixed-block resampling (Künsch). It is **not** the Politis–Romano stationary bootstrap, which uses random block lengths. The adaptation conditions on the observed segments, has finite-sample endpoint effects, and does not restore unknown days or guarantee coverage under regime change. The five-day interval is primary; ten- and twenty-day intervals are sensitivities, never a menu from which to pick significance.

For ES97.5, sort daily losses from largest to smallest and average exactly the worst 2.5% probability mass, fractionally weighting the boundary observation (Acerbi and Tasche). Compute each policy's ES before taking a difference. ES of the paired differences answers another question. The notebook reports actual tail mass, contributing dates and leave-one-day-out sensitivity; it does not manufacture reliable tail inference from a small sample (Yamai and Yoshiba).


In [ ]:
def expected_shortfall(losses, confidence=.975):
    values = np.sort(np.asarray(losses,dtype=float))[::-1]
    if len(values)==0 or not np.isfinite(values).all() or not 0<confidence<1:
        raise ValueError('ES requires a finite nonempty sample and confidence in (0,1)')
    mass = (1-confidence)*len(values)
    weights = np.clip(mass-np.arange(len(values)),0,1)
    return float(np.dot(weights,values)/mass)

def eligible_segments(dates, calendar):
    positions = {str(d):i for i,d in enumerate(calendar)}
    loc = [positions[str(d)] for d in dates]
    if len(set(loc)) != len(loc) or loc != sorted(loc):
        raise ValueError('Eligible dates must be distinct and chronological')
    edges = [0] + [i for i in range(1,len(loc)) if loc[i] != loc[i-1]+1] + [len(loc)]
    return [np.arange(a,b) for a,b in zip(edges[:-1],edges[1:]) if b>a]

def paired_block_interval(wide, calendar, left, right, block_length, spec):
    delta = wide[left].to_numpy()-wide[right].to_numpy()
    segments = eligible_segments(wide.index, calendar)
    lengths = [len(s) for s in segments]
    result = dict(n_days=len(delta),mean_difference_bp=float(np.mean(delta)) if len(delta) else np.nan,
                  block_length=block_length,segment_lengths=str(lengths),
                  support_blocks=sum(n//block_length for n in lengths),lower_bp=np.nan,upper_bp=np.nan)
    if not lengths or min(lengths)<block_length:
        return {**result,'status':'UNAVAILABLE: a segment is shorter than the block length'}
    if result['support_blocks'] < spec['minimum_block_support']:
        return {**result,'status':'UNAVAILABLE: fewer than ten support blocks'}
    matrix = wide[[left,right]].to_numpy(dtype=float)
    if not np.isfinite(matrix).all():
        raise ValueError('Never zero-fill unknown daily outcomes before inference')
    rng = np.random.default_rng(spec['seed'])
    sampled_means = np.empty(spec['repetitions'])
    for j in range(spec['repetitions']):
        indices = []
        for segment in segments:
            starts = rng.integers(0,len(segment)-block_length+1,size=math.ceil(len(segment)/block_length))
            indices.extend(np.concatenate([segment[s:s+block_length] for s in starts])[:len(segment)])
        paired = matrix[np.asarray(indices,dtype=int)]
        sampled_means[j] = np.mean(paired[:,0]-paired[:,1])
    alpha = 1-spec['confidence']
    low, high = np.quantile(sampled_means,[alpha/2,1-alpha/2])
    return {**result,'lower_bp':float(low),'upper_bp':float(high),'status':'AVAILABLE: conditional moving-block percentile interval'}

def tail_rows(series, confidence=.975):
    losses = (-series).sort_values(ascending=False,kind='stable')
    mass = (1-confidence)*len(losses)
    weights = np.clip(mass-np.arange(len(losses)),0,1)
    table = pd.DataFrame({'date':losses.index,'loss_bp':losses.values,'tail_weight_days':weights})
    return table[table.tail_weight_days>0].reset_index(drop=True)

assert abs(expected_shortfall(np.arange(100),.975)-98.2)<1e-12
assert expected_shortfall([2,1],.975)==2
assert abs(expected_shortfall([-5,-2],.975)+2)<1e-12
synthetic_dates = pd.bdate_range('2000-01-03',periods=100).strftime('%Y-%m-%d').tolist()
synthetic_pairs = pd.DataFrame({'a':np.arange(100)/100+2,'b':np.arange(100)/100},index=synthetic_dates)
check_ci = paired_block_interval(synthetic_pairs,synthetic_dates,'a','b',5,INFERENCE)
assert np.allclose([check_ci['lower_bp'],check_ci['upper_bp']],[2,2])
gap_pairs = synthetic_pairs.drop(index=synthetic_dates[2])
assert paired_block_interval(gap_pairs,synthetic_dates,'a','b',5,INFERENCE)['status'].startswith('UNAVAILABLE')
assert np.isclose(expected_shortfall(-synthetic_pairs.a)-expected_shortfall(-synthetic_pairs.b),-2)
print('Inference checks passed: exact fractional tail, negative losses, paired resampling, short-gap withholding.')


## 6. Execute the declared experiment, with a final-test lock

All seven cost/delay settings are run under both timestamp interpretations. Each run applies both candle paths to the same three policies. These are fourteen **scenarios**, not fourteen independent samples or a strategy search. Detailed prices, positions and cash events are written only to the private runtime directory. Funded equity is streamed to disk, so it is not accumulated as millions of Python dictionaries.

Before any strategy replay, the notebook fingerprints the parsed data, scientific functions, parameters, calendar and inference rule. A final run must match a previously saved validation freeze. The freeze is a reproducibility record, not an external preregistration: the timestamp is recorded by the runtime. Source metadata may still limit interpretation even when every software check passes.

**Failure rule.** Any independent-session funding failure is an economic failure, not an ordinary missing-data exclusion. It is retained and prevents a primary superiority conclusion. A funded diagnostic also retains its first failure or unknown-equity event; later independent sessions cannot repair its unknown cash.


In [ ]:
def parsed_data_fingerprint(db_path):
    digest = hashlib.sha256()
    with sqlite3.connect(Path(db_path).resolve().as_uri()+'?mode=ro',uri=True) as con:
        for row in con.execute('SELECT symbol,stamp,o,h,l,c,volume FROM bars ORDER BY symbol,stamp'):
            digest.update((json.dumps(row,separators=(',',':'),allow_nan=False)+'\n').encode())
    return digest.hexdigest()

def read_private_csv(path):
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def run_one_scenario(label, setting):
    scenario = f'{label}__{setting["name"]}'
    private = WORK_DIR / scenario
    private.mkdir()
    equity_state = {'writer':None}
    started = clock.perf_counter()
    with (private/'equity.csv').open('w',newline='',encoding='utf-8') as handle:
        def sink(row):
            if equity_state['writer'] is None:
                equity_state['writer'] = csv.DictWriter(handle,fieldnames=list(row))
                equity_state['writer'].writeheader()
            equity_state['writer'].writerow(row)
        result = run_research(DB_PATH,{**CFG,**setting,'timestamp_label':label},CALENDAR,EVALUATION_DATES,equity_sink=sink)
    for name in ['daily','trades','cash_events','diagnostics','candidates','segments']:
        pd.DataFrame(result[name]).to_csv(private/f'{name}.csv',index=False)
    result['metadata']['elapsed_seconds'] = clock.perf_counter()-started
    (private/'metadata.json').write_text(json.dumps(result['metadata'],indent=2)+'\n')
    # Only small daily/diagnostic tables remain in memory across scenarios.
    return {'daily':pd.DataFrame(result['daily']), 'diagnostics':pd.DataFrame(result['diagnostics']),
            'segments':pd.DataFrame(result['segments']), 'metadata':result['metadata'], 'private_dir':private}

NOTEBOOK_BUNDLE_SHA256 = '89573d0a8d4153e6ef7b56bde048fb07d23d3581c47cb2c9750615c546a70fed'
source_functions = {name:inspect.getsource(value).strip() for name,value in list(globals().items())
                    if inspect.isfunction(value) and value.__module__=='__main__'}
function_hash = hashlib.sha256(json.dumps(source_functions,sort_keys=True).encode()).hexdigest()
data_hash = parsed_data_fingerprint(DB_PATH)
SCIENCE_PAYLOAD = dict(protocol_id='NSE-three-exit-notebook-v2',configuration=CFG,settings=SETTINGS,
    inference=INFERENCE,calendar=[d.isoformat() for d in CALENDAR],splits=split_table.astype(str).to_dict('records'),
    parsed_data_sha256=data_hash,implementation_sha256=function_hash,notebook_bundle_sha256=NOTEBOOK_BUNDLE_SHA256,
    source_reverification=DATA_AUDIT['raw_inputs_reverified'],
    timestamp_semantics='unconfirmed: report both conditional interpretations',
    source_precedence='original monthly delivery; no unreconciled supplementary merge')
PROTOCOL_SHA256 = hashlib.sha256(json.dumps(SCIENCE_PAYLOAD,sort_keys=True).encode()).hexdigest()
FREEZE_PATH = SAVE_BASE / 'protocol_freeze_v2.json'
if RUN_STAGE!='pilot' and EVALUATION_DATES!=PARTITIONS[RUN_STAGE]:
    raise RuntimeError('Evaluation dates differ from the declared chronological partition')
if RUN_STAGE == 'final':
    if EXISTING_DB_PATH or not FREEZE_PATH.is_file():
        raise RuntimeError('Final test locked: a full-data validation freeze is required')
    frozen = json.loads(FREEZE_PATH.read_text())
    if frozen['protocol_sha256'] != PROTOCOL_SHA256:
        raise RuntimeError('Final test locked: data, code, calendar or protocol changed since the freeze')
    with (SAVE_BASE/'final_test_access_log.jsonl').open('a',encoding='utf-8') as log_file:
        log_file.write(json.dumps(dict(run_id=RUN_ID,protocol_sha256=PROTOCOL_SHA256,
                                      access_utc=datetime.now(timezone.utc).isoformat()))+'\n')
    print('Matching freeze verified. Final-test access is recorded; this period is now observed.')
else:
    print('Final-test outcomes are not accessed in this stage.')
print('Protocol SHA-256:',PROTOCOL_SHA256)
RUNS = {}
for label in ['start','end']:
    for setting in SETTINGS:
        RUNS[(label,setting['name'])] = run_one_scenario(label,setting)
        current = RUNS[(label,setting['name'])]
        common = current['daily'].query("accounting == 'independent' and status == 'EVALUABLE'").date.nunique()
        print(f'{label:5} {setting["name"]:18} | {common:4} evaluable days | {current["metadata"]["elapsed_seconds"]:.1f}s')


## 7. What actually entered the estimates?

Every scheduled date remains in the diagnostic ledger. A valid no-signal day contributes a zero return. A date with unavailable candidate information does not. Funded accounts can abstain on information known to be unavailable before entry; their day count can therefore differ from the primary complete-case count. Compare the denominators before comparing statistics.

The code below checks that all six policy/path primary series use the same eligible dates. An execution, funding or data failure is counted before any numeric summary is formed. Failed sessions are never replaced by zero. The printed development candidate is a trace of an observed signal; it is not a second independent observation or evidence about final-test opportunities.


In [ ]:
ALL_DAILY = pd.concat([run['daily'].assign(timestamp_label=label,setting=name)
                      for (label,name),run in RUNS.items()],ignore_index=True)
ALL_DAILY['return_bp'] = ALL_DAILY.return_on_reference_capital*10000
status_table = (ALL_DAILY.groupby(['timestamp_label','setting','accounting','status'])
                .date.nunique().rename('scheduled_days').reset_index())
display(status_table[status_table.setting.eq('reference')])
ECONOMIC_FAILURE = bool(((ALL_DAILY.accounting=='independent') & ALL_DAILY.status.eq('FUNDING_FAILURE')).any())
print('Independent-session economic failure:',ECONOMIC_FAILURE)
for key, run in RUNS.items():
    good = run['daily'].query("accounting == 'independent' and status == 'EVALUABLE'")
    sets = [set(g.date) for _,g in good.groupby(['policy','path'])]
    assert not sets or (len(sets)==6 and all(s==sets[0] for s in sets))
candidate_example = read_private_csv(RUNS[('start','reference')]['private_dir']/'candidates.csv')
if not candidate_example.empty:
    fields = ['date','symbol','decision_at','stock_return','relative_return','prior_relative_mean',
              'prior_relative_sd','stock_volatility','z']
    display(candidate_example[fields].head(1))
    first = candidate_example.iloc[0]
    assert np.isclose(first.z,(first.relative_return-first.prior_relative_mean)/first.prior_relative_sd)
    day_position = CALENDAR.index(date.fromisoformat(first.date))
    print('Exactly preceding scheduled dates:',[str(d) for d in CALENDAR[max(0,day_position-20):day_position]])
else:
    print('No eligible market signal in this scenario; do not invent a worked market example.')


## 8. Returns, paired effects and exit mechanisms

Policy means are daily **P&L / fixed reference capital**, expressed in basis points. They are not returns on changing wealth, and are not compounded. Mean effects compare paired days; repeated path scenarios are displayed separately. The primary question is whether the combined policy improves on the fixed-time policy. A stop-only result does not answer that primary question.

Exit-reason counts and uniquely identified both-hit episodes help explain any difference. Durations from within-minute exits are intervals, not observed execution seconds. No ambiguous episodes is an informative zero within the sampled design, not a reason to change barriers until ambiguity appears.


In [ ]:
MEANS = (ALL_DAILY.query("accounting == 'independent' and status == 'EVALUABLE'")
         .groupby(['timestamp_label','setting','policy','path']).return_bp.agg(['count','mean','std']).reset_index()
         .rename(columns={'count':'evaluable_days','mean':'mean_daily_pnl_bp','std':'daily_sd_bp'}))
CONTRASTS = [('combined_minus_time','stop_and_target','time_only'),
             ('stop_minus_time','stop_only','time_only'),('combined_minus_stop','stop_and_target','stop_only')]
PAIRED = []
for (label,name), run in RUNS.items():
    good = run['daily'].query("accounting == 'independent' and status == 'EVALUABLE'").copy()
    good['bp'] = good.return_on_reference_capital*10000
    for path,group in good.groupby('path'):
        wide = group.pivot(index='date',columns='policy',values='bp').sort_index()
        assert wide.notna().all().all()
        for contrast,left,right in CONTRASTS:
            PAIRED.append(dict(timestamp_label=label,setting=name,path=path,contrast=contrast,n_days=len(wide),
                               mean_difference_bp=(wide[left]-wide[right]).mean()))
PAIRED = pd.DataFrame(PAIRED,columns=['timestamp_label','setting','path','contrast','n_days','mean_difference_bp'])
display(MEANS[MEANS.setting.eq('reference')].round(4))
display(PAIRED[PAIRED.setting.eq('reference')].round(4))
mechanism_records = []
for label in ['start','end']:
    trades = read_private_csv(RUNS[(label,'reference')]['private_dir']/'trades.csv')
    if trades.empty:
        mechanism_records.append(dict(timestamp_label=label,unique_episodes=0,ambiguous_episodes=0,ambiguity_rate=np.nan))
        continue
    kept = trades[(trades.accounting=='independent') & trades.status.eq('RESOLVED') & trades.included_primary_day.eq(True)]
    if kept.empty:
        print(label, ': no resolved episodes on complete-case dates; ambiguity rate unavailable.')
        mechanism_records.append(dict(timestamp_label=label,unique_episodes=0,ambiguous_episodes=0,ambiguity_rate=np.nan))
        continue
    counts = kept.groupby(['policy','path','exit_reason']).size().rename('executions').reset_index()
    display(counts.assign(timestamp_label=label))
    both = kept[kept.ambiguous.eq(True)].episode_id.nunique()
    unique = kept.episode_id.nunique()
    mechanism_records.append(dict(timestamp_label=label,unique_episodes=unique,ambiguous_episodes=both,
                                   ambiguity_rate=both/unique if unique else np.nan))
MECHANISMS = pd.DataFrame(mechanism_records,columns=['timestamp_label','unique_episodes','ambiguous_episodes','ambiguity_rate'])
display(MECHANISMS)
fig, ax = plt.subplots(figsize=(8,3.5))
ref_plot = PAIRED.query("setting == 'reference' and path == 'O-L-H-C'")
if not ref_plot.empty:
    ref_plot.pivot(index='contrast',columns='timestamp_label',values='mean_difference_bp').rename(index={
        'combined_minus_stop':'Combined − stop','combined_minus_time':'Combined − time',
        'stop_minus_time':'Stop − time'}).plot.bar(ax=ax,rot=0)
    ax.legend(title='Minute labels')
ax.axhline(0,color='black',lw=.8)
ax.set(ylabel='Paired daily P&L difference\n(bp of reference capital)',xlabel='Exit-policy contrast',
       title=f'{RUN_STAGE}: reference execution, O-L-H-C')
ax.tick_params(axis='x',labelsize=8); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'paired_effects.png'); plt.show()


## 9. Funded equity and drawdown — a separate diagnostic

The funded account carries realised P&L between days, retains fixed entry notionals, records every entry fee and exit cash flow, and values held shares at minute closes. Realised early-exit proceeds are part of equity but unavailable for new entries until common expiry. No deposit repairs a loss or unknown outcome within a segment. Each evaluation partition starts separately with C₀.

The engine evaluates each minute close and retains changes plus boundary points; omitting identical consecutive values does not alter sampled drawdown. Here $DD_t=1-E_t/\max_{u\le t}E_u$. Minute-close drawdown is not the worst within-minute drawdown. The graph is downsampled for display, but maxima are computed from **every saved mark**. It never joins different segments. A terminated funded prefix has an outcome-dependent endpoint and cannot replace the primary fixed-calendar estimate.


In [ ]:
FUNDED_RISK, PLOT_EQUITY = [], []
for label in ['start','end']:
    path = RUNS[(label,'reference')]['private_dir']/'equity.csv'
    maxima, samples, counters = {}, {}, {}
    if path.stat().st_size == 0:
        continue
    for chunk in pd.read_csv(path,chunksize=25000):
        for key, group in chunk.groupby(['segment','policy','path'],sort=False):
            old = maxima.setdefault(key,dict(max_drawdown=0.0,marks=0,first_date=str(group.date.iloc[0]),last_date=None))
            old['max_drawdown'] = max(old['max_drawdown'],float(group.drawdown_fraction.max()))
            old['marks'] += len(group); old['last_date'] = str(group.date.iloc[-1])
            count = counters.get(key,0)
            display_rows = group.iloc[np.where((np.arange(len(group))+count)%25==0)[0]]
            # Always retain the last mark of this chunk; rendering only, not the risk computation.
            display_rows = pd.concat([display_rows,group.tail(1)]).drop_duplicates('timestamp')
            samples.setdefault(key,[]).append(display_rows[['timestamp','equity','drawdown_fraction']])
            counters[key] = count+len(group)
    for key,value in maxima.items():
        FUNDED_RISK.append(dict(timestamp_label=label,segment=key[0],policy=key[1],path=key[2],
                                max_drawdown_pct=100*value['max_drawdown'],saved_marks=value['marks'],
                                first_date=value['first_date'],last_date=value['last_date']))
        sample = pd.concat(samples[key]).drop_duplicates('timestamp').sort_values('timestamp')
        PLOT_EQUITY.append((label,key,sample))
FUNDED_RISK = pd.DataFrame(FUNDED_RISK)
display(FUNDED_RISK.round(4))
fig,axes = plt.subplots(2,2,figsize=(12,6),sharex='col')
for label,key,frame in PLOT_EQUITY:
    if key[2]!='O-L-H-C':
        continue
    j = ['start','end'].index(label)
    ts = pd.to_datetime(frame.timestamp,utc=True)
    axes[0,j].plot(ts,frame.equity,label=f'{key[1].replace("_"," ")} · segment {key[0]}')
    axes[1,j].plot(ts,100*frame.drawdown_fraction)
for j,label in enumerate(['start','end']):
    axes[0,j].set(title=f'{label} labels · funded reference O-L-H-C',ylabel='Marked equity (INR)')
    if axes[0,j].lines: axes[0,j].legend(fontsize=7)
    axes[1,j].set(ylabel='Sampled drawdown (%)',xlabel='Time (UTC; source assumed IST)')
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'funded_equity_drawdown.png'); plt.show()
for label in ['start','end']:
    display(RUNS[(label,'reference')]['segments'].assign(timestamp_label=label))


## 10. Expected shortfall and statistical uncertainty

ES below uses the same independent-session complete-case set as the primary means. It does not silently switch to the different funded-abstention sample. Loss is minus daily P&L / C₀; ES may be negative if even the worst sampled days are profitable. Tail-date ties are ordered only to present a fractional empirical average, not to claim a unique extreme-day attribution.

Intervals are conditional on the observed sample and implemented resampling assumptions. They do not repair selective exclusions or uncertain prices. A 95% interval **containing zero, including an endpoint**, is inconclusive for superiority. A negative combined-minus-time effect is not reframed as improvement merely because some secondary statistic is lower.


In [ ]:
ES_ROWS, TAILS, INTERVALS = [], [], []
for label in ['start','end']:
    good = ALL_DAILY.query("timestamp_label == @label and setting == 'reference' and accounting == 'independent' and status == 'EVALUABLE'")
    for path,group in good.groupby('path'):
        wide = group.pivot(index='date',columns='policy',values='return_bp').sort_index()
        for policy in POLICIES:
            values = wide[policy]
            estimate = expected_shortfall(-values)
            leave_one = [expected_shortfall(-values.drop(index=d)) for d in values.index] if len(values)>1 else [np.nan]
            ES_ROWS.append(dict(timestamp_label=label,path=path,policy=policy,n_days=len(values),
                                tail_mass_days=.025*len(values),ES975_loss_bp=estimate,
                                leave_one_day_out_min_bp=np.nanmin(leave_one),leave_one_day_out_max_bp=np.nanmax(leave_one)))
            TAILS.append(tail_rows(values).assign(timestamp_label=label,path=path,policy=policy))
        for contrast,left,right in CONTRASTS:
            for b in INFERENCE['block_lengths']:
                interval = paired_block_interval(wide,[d.isoformat() for d in EVALUATION_DATES],left,right,b,INFERENCE)
                if ECONOMIC_FAILURE:
                    interval.update(status='UNAVAILABLE: independent-account funding failure',lower_bp=np.nan,upper_bp=np.nan)
                INTERVALS.append(dict(timestamp_label=label,path=path,contrast=contrast,**interval))
ES_TABLE = pd.DataFrame(ES_ROWS)
TAIL_TABLE = pd.concat(TAILS,ignore_index=True) if TAILS else pd.DataFrame()
INTERVAL_TABLE = pd.DataFrame(INTERVALS)
display(ES_TABLE.query("path == 'O-L-H-C'").round(4) if not ES_TABLE.empty else ES_TABLE)
display(TAIL_TABLE.head(18))
display(INTERVAL_TABLE.query("path == 'O-L-H-C' and contrast == 'combined_minus_time'") if not INTERVAL_TABLE.empty else INTERVAL_TABLE)
if not ES_TABLE.empty:
    es_wide = ES_TABLE.pivot(index=['timestamp_label','path'],columns='policy',values='ES975_loss_bp')
    es_wide['combined_minus_time_ES_bp'] = es_wide.stop_and_target-es_wide.time_only
    es_wide['stop_minus_time_ES_bp'] = es_wide.stop_only-es_wide.time_only
    display(es_wide.round(4))


## 11. Execution sensitivity and sample alignment

Costs, extra stop slippage and delays are changed one at a time. Identical round-trip charges cancel from paired completed-trade contrasts under this fixed-entry-notional charge model; they still change absolute policy returns. Stop penalties can change rankings because only some policies stop. Actual fees based on exit value need not cancel. Longer entry delays reanchor the entry and both barriers; they are not a subtraction of an arbitrary penalty after the trade.

The main tables retain each scenario's own common evaluable sample. The table below additionally aligns reference and stress dates within each timestamp interpretation. This exposes how much a sensitivity comparison depends on its sample. Timestamp assumptions can change the signal itself and should not be selected by which gives a better return. The costs are scenario values, not a claim about historical Indian brokerage or market impact.


In [ ]:
ALIGNMENT = []
for label in ['start','end']:
    part = ALL_DAILY.query("timestamp_label == @label and accounting == 'independent' and status == 'EVALUABLE' and path == 'O-L-H-C'")
    ref = part[part.setting.eq('reference')].pivot(index='date',columns='policy',values='return_bp')
    for setting in SETTINGS:
        current = part[part.setting.eq(setting['name'])].pivot(index='date',columns='policy',values='return_bp')
        days = sorted(set(ref.index)&set(current.index))
        if not days:
            continue
        for contrast,left,right in CONTRASTS:
            baseline = (ref.loc[days,left]-ref.loc[days,right]).mean()
            stressed = (current.loc[days,left]-current.loc[days,right]).mean()
            ALIGNMENT.append(dict(timestamp_label=label,setting=setting['name'],contrast=contrast,
                                  aligned_days=len(days),reference_difference_bp=baseline,
                                  scenario_difference_bp=stressed,change_from_reference_bp=stressed-baseline))
ALIGNMENT = pd.DataFrame(ALIGNMENT)
display(ALIGNMENT[ALIGNMENT.contrast.eq('combined_minus_time')].round(4) if not ALIGNMENT.empty else ALIGNMENT)
for name in ['cost_5','cost_25']:
    aligned = ALIGNMENT[ALIGNMENT.setting.eq(name)] if not ALIGNMENT.empty else ALIGNMENT
    if not aligned.empty:
        assert np.allclose(aligned.change_from_reference_bp,0,atol=1e-8)
fig,ax = plt.subplots(figsize=(9,3.4))
if not ALIGNMENT.empty:
    ALIGNMENT[ALIGNMENT.contrast.eq('combined_minus_time')].pivot(index='setting',columns='timestamp_label',values='scenario_difference_bp').plot.bar(ax=ax,rot=25)
    ax.legend(title='Minute labels')
ax.axhline(0,color='black',lw=.8); ax.set(ylabel='Combined − time mean effect\n(bp of reference capital)',xlabel='Declared execution scenario',
                                       title=f'{RUN_STAGE}: one-at-a-time sensitivity, O-L-H-C')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'execution_sensitivity.png'); plt.show()


## 12. What can be concluded from this run?

The text below is generated from the current tables. It deliberately separates a numerical effect from sufficient evidence of superiority, and conditional development results from final-test evidence. Software tests establish arithmetic and event rules; only appropriately qualified market analysis can address the research question.


In [ ]:
conclusion_lines = [f'**Executed stage: {RUN_STAGE}.**',
                    f'The requested input period is {CFG["start_date"]} to {CFG["end_date"]}.',
                    'The primary estimand conditions on common complete-case sessions; omitted sessions are not zero returns.']
if ECONOMIC_FAILURE:
    conclusion_lines.append('**Primary superiority inference is withheld: an independent-session account encountered an economic funding failure.**')
    conclusion_lines.append('Any displayed surviving-day mean is descriptive and does not identify the planned fixed-calendar primary effect after that economic exclusion.')
for label in ['start','end']:
    row = INTERVAL_TABLE.query("timestamp_label == @label and path == 'O-L-H-C' and contrast == 'combined_minus_time' and block_length == 5") if not INTERVAL_TABLE.empty else pd.DataFrame()
    if row.empty:
        conclusion_lines.append(f'Under {label} labels, there are no common days supporting the primary comparison.')
        continue
    row = row.iloc[0]
    line = f'Under {label} labels, combined minus time is {row.mean_difference_bp:.4f} bp per evaluable day (n={row.n_days}). '
    if pd.isna(row.lower_bp):
        line += 'The primary interval is unavailable under the declared support/failure rules; superiority is unestablished.'
    elif row.lower_bp>0:
        line += f'The conditional 95% interval is [{row.lower_bp:.4f}, {row.upper_bp:.4f}] bp and lies above zero.'
    elif row.upper_bp<0:
        line += f'The conditional 95% interval is [{row.lower_bp:.4f}, {row.upper_bp:.4f}] bp and lies below zero.'
    else:
        line += f'The conditional 95% interval is [{row.lower_bp:.4f}, {row.upper_bp:.4f}] bp and contains zero.'
    conclusion_lines.append(line)
if not MECHANISMS.empty:
    conclusion_lines.append('Observed unique ambiguous episodes: '+', '.join(f'{r.timestamp_label} labels {int(r.ambiguous_episodes)}/{int(r.unique_episodes)}' for r in MECHANISMS.itertuples())+'.')
if RUN_STAGE!='final':
    conclusion_lines.append('**These are not final-test conclusions. The final test was not run.**')
conclusion_lines.append('ES is descriptive. Report its tail mass and influential dates; sampled drawdown refers to identified funded segments only. No result verifies actual spread, depth, queue position or executable profitability.')
CONCLUSION = '\n\n'.join(conclusion_lines)
display(Markdown(CONCLUSION))


## 13. Freeze, save and reproduce

After reviewing validation, set `READY_TO_FREEZE=True` **in the save cell below**, then rerun that cell alone. It records the exact protocol identity without rerunning the experiment or overwriting an earlier incompatible freeze. Repeat export is permitted only for the same run ID and protocol. A later `final` run must match that identity. Changing research code or source prices requires a documented new version and invalidates the old freeze. A final-access log prevents a repeat run from being described as a first untouched test.

The exported ZIP contains **aggregate research outputs**, figures and provenance. It excludes source archives, raw price tables and detailed private trade ledgers. The HTML report is a readable companion to this notebook. Save the notebook itself through Colab to preserve its visible cell outputs. For course submission, upload the required report PDF separately from supporting files; saving a notebook does not submit the assignment.


In [ ]:
READY_TO_FREEZE = False # @param {type:"boolean"}
import base64, html, zipfile

tables = {'calendar_split':split_table,'coverage_monthly':coverage_table,'coverage_daily':coverage_daily,
          'status_counts':status_table,'policy_means':MEANS,'paired_effects':PAIRED,
          'exit_ambiguity':MECHANISMS,'funded_drawdown':FUNDED_RISK,'expected_shortfall':ES_TABLE,
          'tail_dates':TAIL_TABLE,'intervals':INTERVAL_TABLE,'aligned_sensitivity':ALIGNMENT}
for name,table in tables.items():
    table.to_csv(OUTPUT_DIR/f'{name}.csv',index=False)
manifest = dict(run_id=RUN_ID,stage=RUN_STAGE,completed_utc=datetime.now(timezone.utc).isoformat(),
                protocol_sha256=PROTOCOL_SHA256,science=SCIENCE_PAYLOAD,
                python=sys.version,packages={m.__name__:m.__version__ for m in [np,pd]},
                independent_economic_failure=ECONOMIC_FAILURE,
                source_status=DATA_AUDIT['mode'],raw_inputs_reverified=DATA_AUDIT['raw_inputs_reverified'],
                source_files=len(DATA_AUDIT.get('input_files',[])),
                caveat='A locally verified cached pilot does not establish a successful mounted-Drive/full-study run.')
try:
    import resource
    manifest['process_peak_rss_mib'] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/(1024 if sys.platform!='darwin' else 1024**2)
except ImportError:
    manifest['process_peak_rss_mib'] = None
(OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str)+'\n',encoding='utf-8')
(OUTPUT_DIR/'conclusions.md').write_text(CONCLUSION+'\n',encoding='utf-8')
# Source provenance retains checksums/member names without account-specific root paths.
public_audit = json.loads(json.dumps(DATA_AUDIT))
for item in public_audit.get('input_files',[]):
    if 'source' in item: item['source'] = Path(item['source']).name
public_audit['excluded_supplements'] = [Path(p).name for p in public_audit.get('excluded_supplements',[])]
(OUTPUT_DIR/'input_audit.json').write_text(json.dumps(public_audit,indent=2)+'\n',encoding='utf-8')
parts = ['<!doctype html><meta charset="utf-8"><title>NSE research run</title>',
         '<style>body{font:16px Georgia;max-width:1100px;margin:40px auto;padding:0 20px;color:#20252c}table{border-collapse:collapse;font:12px Arial;width:100%;margin:20px 0}th,td{padding:6px;border-bottom:1px solid #ddd;text-align:right}img{max-width:100%}pre{white-space:pre-wrap}h1,h2{font-family:Arial}</style>',
         '<h1>'+html.escape(CFG['study_title'])+'</h1>', '<p>Group 17838 · '+html.escape(RUN_ID)+' · '+RUN_STAGE+'</p>',
         '<h2>Evidence and interpretation</h2><pre>'+html.escape(CONCLUSION)+'</pre>']
for name in ['policy_means','paired_effects','status_counts','exit_ambiguity','expected_shortfall','funded_drawdown','intervals']:
    parts.extend(['<h2>'+name.replace('_',' ').title()+'</h2>',tables[name].to_html(index=False,float_format=lambda x:f'{x:.5f}')])
for path in sorted(OUTPUT_DIR.glob('*.png')):
    parts.append('<h2>'+path.stem.replace('_',' ').title()+'</h2><img alt="'+html.escape(path.stem)+'" src="data:image/png;base64,'+base64.b64encode(path.read_bytes()).decode()+'">')
parts.append('<p>Protocol SHA-256: '+PROTOCOL_SHA256+'</p><p>Read the notebook for assumptions, equations, tests, limitations and cited references. No course submission is performed.</p>')
(OUTPUT_DIR/'research_report.html').write_text('\n'.join(parts),encoding='utf-8')
if READY_TO_FREEZE:
    if RUN_STAGE!='validation' or EXISTING_DB_PATH or ECONOMIC_FAILURE:
        raise RuntimeError('Freeze requires a raw-data validation run without independent-account funding failure')
    current_functions = {name:inspect.getsource(value).strip() for name,value in list(globals().items())
                         if inspect.isfunction(value) and value.__module__=='__main__'}
    if (hashlib.sha256(json.dumps(current_functions,sort_keys=True).encode()).hexdigest()!=function_hash
        or parsed_data_fingerprint(DB_PATH)!=data_hash
        or hashlib.sha256(json.dumps(SCIENCE_PAYLOAD,sort_keys=True).encode()).hexdigest()!=PROTOCOL_SHA256):
        raise RuntimeError('Code, data or protocol changed after execution; do not freeze stale results')
    freeze = dict(protocol_sha256=PROTOCOL_SHA256,science=SCIENCE_PAYLOAD,
                  validation_run_id=RUN_ID,created_utc=datetime.now(timezone.utc).isoformat(),
                  caveat='Internal version record, not an independently timestamped preregistration')
    if FREEZE_PATH.exists():
        if json.loads(FREEZE_PATH.read_text())['protocol_sha256'] != PROTOCOL_SHA256:
            raise RuntimeError('Existing freeze differs. Preserve it and document an explicit protocol amendment.')
        print('Matching freeze already exists; preserved.')
    else:
        with FREEZE_PATH.open('x',encoding='utf-8') as handle:
            json.dump(freeze,handle,indent=2)
        print('Validation protocol frozen:',FREEZE_PATH)
SAVE_DIR = SAVE_BASE/RUN_ID
if SAVE_DIR.exists():
    previous = json.loads((SAVE_DIR/'run_manifest.json').read_text())
    if previous['protocol_sha256']!=PROTOCOL_SHA256:
        raise RuntimeError('Existing run folder has a different protocol; preserve it')
shutil.copytree(OUTPUT_DIR,SAVE_DIR,dirs_exist_ok=True)
zip_path = SAVE_BASE/(RUN_ID+'_research_outputs.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as package:
    for file in sorted(SAVE_DIR.iterdir()): package.write(file,arcname=file.name)
print('Saved research outputs:',SAVE_DIR)
print('Supporting ZIP:',zip_path)
print('Final test evaluated:',RUN_STAGE=='final')


## 14. Limits and the contribution this experiment can make

The contribution is a controlled, reproducible comparison of the submitted exits under explicit observational and execution limits. It measures whether findings depend on the chosen path, cost, delay, timestamp interpretation and missing-data rules. It does not establish a new theorem about stops, calibrate an optimal stopping model, or certify a deployable strategy.

* **Data identity:** timestamp semantics, adjustment conventions and execution-relevant metadata require supplier confirmation. The original monthly delivery is used; conflicting supplements remain outside the analysis until reconciled independently of profit.
* **Selection and exclusions:** five fixed names are a case study. Complete-case dates can be selected by market conditions. An unavailable day is not known to have zero return. A retrospective outage exclusion is not a lower bound on losses.
* **Execution:** OHLC does not reveal queue priority, executable spread or capacity. Stop and target touches are conditional fills; delay scenarios and the two extreme-order paths do not exhaust execution risk.
* **Accounting:** fractional shares, fixed notionals, no borrowing and blocked reuse are research assumptions. The funded path is separate from independent-session effects. Unknown losses are not erased by an automatic reset.
* **Inference:** the segment-aware bootstrap is an explicit approximation, not a guarantee under nonstationarity. Few tail events limit ES precision. Scenario runs are paired, not independent replications. PBO, Reality Check and Deflated Sharpe are not claimed without their required selection experiments (Sullivan et al.; Bailey et al.).
* **Integrity:** negative and inconclusive results remain valid outcomes. Any change after final-test inspection must be labelled exploratory and logged. The final test's role cannot be restored by renaming a notebook.

### References

Acerbi, Carlo, and Dirk Tasche. “On the Coherence of Expected Shortfall.” *arXiv*, version 5, 2 May 2002, https://arxiv.org/abs/cond-mat/0104295v5.

Bailey, David H., et al. “The Probability of Backtest Overfitting.” *Journal of Computational Finance*, vol. 20, no. 4, 2017, pp. 39–69, https://doi.org/10.21314/JCF.2016.322.

Kaminski, Kathryn M., and Andrew W. Lo. “When Do Stop-Loss Rules Stop Losses?” *Journal of Financial Markets*, vol. 18, 2014, pp. 234–54, https://doi.org/10.1016/j.finmar.2013.07.001.

Künsch, Hans R. “The Jackknife and the Bootstrap for General Stationary Observations.” *The Annals of Statistics*, vol. 17, no. 3, 1989, pp. 1217–41, https://doi.org/10.1214/aos/1176347265.

Leung, Tim, and Xin Li. “Optimal Mean Reversion Trading with Transaction Costs and Stop-Loss Exit.” *arXiv*, version 3, 14 May 2015, https://arxiv.org/abs/1411.5062v3.

Maier-Paape, Stanislaus, and Andreas Platen. “Backtest of Trading Systems on Candle Charts.” *IFTA Journal*, 2016, pp. 10–17, https://www.ifta.org/assets/docs/d_ifta_journal_16.pdf.

National Stock Exchange of India. *Trading Holidays for the Calendar Year 2021*. Circular NSE/CMTR/46623, 11 Dec. 2020, https://archives.nseindia.com/content/circulars/CMTR46623.pdf.

National Stock Exchange of India. *Trading Holidays for the Calendar Year 2022*. Circular NSE/CMTR/50560, 10 Dec. 2021, https://archives.nseindia.com/content/circulars/CMTR50560.pdf.

National Stock Exchange of India. *Trading Holidays for the Calendar Year 2023*. Circular NSE/CMTR/54757, 8 Dec. 2022, https://archives.nseindia.com/content/circulars/CMTR54757.pdf.

NSE Clearing Limited. *Changes in Settlement Schedule on Account of Holiday on June 29, 2023*. Circular NCL/CMPT/57291, 27 June 2023, https://archives.nseindia.com/content/circulars/CMPT57291.pdf. References trading-holiday amendment NSE/CMTR/57285.

Politis, Dimitris N., and Joseph P. Romano. “The Stationary Bootstrap.” *Journal of the American Statistical Association*, vol. 89, no. 428, 1994, pp. 1303–13, https://doi.org/10.1080/01621459.1994.10476870.

Securities and Exchange Board of India. *Adjudication Order in Respect of National Stock Exchange of India Limited*. 20 June 2023, https://www.sebi.gov.in/sebi_data/attachdocs/jun-2023/1687270559560.pdf.

Sullivan, Ryan, et al. *Data-Snooping, Technical Trading Rule Performance, and the Bootstrap*. Financial Markets Group, London School of Economics, Oct. 1998, Discussion Paper 303, https://www.fmg.ac.uk/sites/default/files/2020-11/dp303.pdf.

Yamai, Yasuhiro, and Toshinao Yoshiba. “Comparative Analyses of Expected Shortfall and Value-at-Risk: Their Estimation Error, Decomposition, and Optimization.” *Monetary and Economic Studies*, vol. 20, no. 1, 2002, pp. 87–122, https://www.imes.boj.or.jp/research/papers/english/me20-1-4.pdf.
